# V2a: EEGNet-Inspired Tri-Contrastive Visual Imagery Decoding

**Key changes from v1:**
- Model reduced from EEGNet-32,2 to EEGNet-8,2 (primary) and EEGNet-4,2 (comparison)
- FBCSP baseline (9 bands, mutual-information feature selection) replaces broadband CSP
- Dense CE baseline (imagery-only, pure cross-entropy, no contrastive loss)
- Optuna search subjects selected by median FBCSP performance, then **excluded** from all evaluation
- Dual ablation: frontal-7 zero + posterior-17 retain
- ICA caching for dramatically reduced runtime
- Purged temporal block CV preserved from v1

**Pipeline:**
1. Preprocessing + ICA (cached)
2. FBCSP baselines on all 22 subjects -> rank -> select 3 median for Optuna
3. Optuna HP search on 3 search subjects (excluded from all downstream results)
4. Evaluate on remaining 19 subjects: Tri-Contrastive (8,2 and 4,2), Dense CE, FBCSP
5. Ablation studies (frontal-7, posterior-17)
6. Cross-session generalization
7. Confusion matrices, test-retest, statistics


In [2]:
import sys, json, time, hashlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from scipy import stats
from scipy.signal import butter, sosfilt
from mne.decoding import CSP
import mne

warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')

PROJECT_ROOT = Path('.').resolve()
DATASET_DIR = PROJECT_ROOT / 'dataset'
OUTPUT_DIR = PROJECT_ROOT / 'outputs_v2a'
ICA_CACHE_DIR = PROJECT_ROOT / 'ica_cache'
OUTPUT_DIR.mkdir(exist_ok=True)
ICA_CACHE_DIR.mkdir(exist_ok=True)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

TASKS = {
    'AVI': {'n_classes': 3, 'chance': 1/3, 'classes': ['bird', 'dog', 'fish']},
    'FVI': {'n_classes': 3, 'chance': 1/3, 'classes': ['circle', 'pentagram', 'square']},
    'OVI': {'n_classes': 4, 'chance': 1/4, 'classes': ['chair', 'cup', 'scissors', 'watch']},
}

N_FOLDS = 5
CROP_LENGTH = 625
EVAL_EPOCHS = 150
EVAL_PATIENCE = 25
SEARCH_EPOCHS = 80
SEARCH_PATIENCE = 15
SEARCH_N_FOLDS = 3
SEARCH_TASK = 'FVI'
N_OPTUNA_TRIALS = 50

CHANNEL_NAMES_32 = [
    'Fpz','Fp1','Fp2','Fz','F3','F4','F7','F8','FCz','FC3','FC4',
    'FT7','FT8','Cz','C3','C4','T7','T8','CP3','CP4','TP7','TP8',
    'Pz','P3','P4','P7','P8','PO3','PO4','Oz','O1','O2'
]
FRONTAL_ABLATION_CHANNELS = ['Fp1', 'Fp2', 'Fpz', 'F7', 'F8', 'FT7', 'FT8']

# Posterior retention set (19 channels). Drop T7/T8 for strict 17 if preferred.
POSTERIOR_RETAIN_CHANNELS = [
    'Cz', 'C3', 'C4', 'T7', 'T8',
    'CP3', 'CP4', 'TP7', 'TP8',
    'Pz', 'P3', 'P4', 'P7', 'P8',
    'PO3', 'PO4', 'Oz', 'O1', 'O2'
]

subjects_all = sorted([d.name for d in DATASET_DIR.iterdir()
                        if d.is_dir() and d.name.startswith('sub-')])
print(f'{len(subjects_all)} total subjects: {subjects_all}')


Device: cuda
22 total subjects: ['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10', 'sub-11', 'sub-12', 'sub-13', 'sub-14', 'sub-15', 'sub-16', 'sub-17', 'sub-18', 'sub-19', 'sub-20', 'sub-21', 'sub-22']


In [3]:
# ============================================================
# Data Loading with ICA Caching
# ============================================================

def load_and_preprocess_cached(bdf_path, apply_ica=True, ica_confidence=0.80):
    """Load BDF, bandpass, CAR, ICA artifact rejection with disk caching."""
    bdf_path = Path(bdf_path)
    
    if apply_ica:
        cache_key = hashlib.md5(
            f'{bdf_path.resolve()}|{ica_confidence}'.encode()
        ).hexdigest()[:12]
        cache_fif = ICA_CACHE_DIR / f'{bdf_path.stem}_{cache_key}_raw.fif'
        cache_meta = ICA_CACHE_DIR / f'{bdf_path.stem}_{cache_key}_meta.json'
        
        if cache_fif.exists() and cache_meta.exists():
            raw = mne.io.read_raw_fif(cache_fif, preload=True)
            with open(cache_meta) as f:
                meta = json.load(f)
            return raw, meta['n_removed']
    
    raw = mne.io.read_raw_bdf(bdf_path, preload=True)
    raw.pick_types(eeg=True, stim=False)
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage, on_missing='warn')
    raw.filter(l_freq=1.0, h_freq=40.0, method='iir',
               iir_params={'order': 4, 'ftype': 'butter'})
    raw.set_eeg_reference('average', projection=False)
    
    n_removed = 0
    if apply_ica:
        try:
            from mne_icalabel import label_components
        except ImportError:
            print(f'WARNING: mne_icalabel not installed, skipping ICA')
            return raw, 0
        
        ica = mne.preprocessing.ICA(
            n_components=min(20, len(raw.ch_names) - 1),
            method='fastica', random_state=42, max_iter=500)
        ica.fit(raw)
        ic_labels = label_components(raw, ica, method='iclabel')
        
        exclude = []
        for i, (label, prob) in enumerate(zip(ic_labels['labels'],
                                               ic_labels['y_pred_proba'])):
            if label in ('eye blink', 'muscle artifact') and float(prob) > ica_confidence:
                exclude.append(i)
        if exclude:
            ica.exclude = exclude
            ica.apply(raw)
            n_removed = len(exclude)
        
        raw.save(cache_fif, overwrite=True)
        with open(cache_meta, 'w') as f:
            json.dump({'n_removed': n_removed, 'bdf_path': str(bdf_path)}, f)
        print(f'  ICA cached: {bdf_path.name} (removed {n_removed})')
    
    return raw, n_removed


def extract_epochs_from_block(raw, events_df, sfreq=1000, target_sfreq=250):
    """Extract perception and imagery epochs for all classes."""
    class_names = sorted(events_df['type'].unique())
    class_map = {name: i for i, name in enumerate(class_names)}
    perc_list, img_list, label_list = [], [], []
    
    for _, row in events_df.iterrows():
        imagery_onset = int(row['onset'] * sfreq)
        imagery_end = imagery_onset + int(4.0 * sfreq)
        perception_onset = int((row['onset'] - 6.0) * sfreq)
        perception_end = perception_onset + int(4.0 * sfreq)
        baseline_start = perception_onset - int(1.0 * sfreq)
        baseline_end = perception_onset
        
        if baseline_start < 0 or imagery_end > raw.n_times:
            continue
        
        baseline_mean = raw.get_data(start=baseline_start, stop=baseline_end).mean(axis=1, keepdims=True)
        perc = raw.get_data(start=perception_onset, stop=perception_end) - baseline_mean
        img = raw.get_data(start=imagery_onset, stop=imagery_end) - baseline_mean
        perc_list.append(perc)
        img_list.append(img)
        label_list.append(class_map[row['type']])
    
    perception, imagery, labels = np.array(perc_list), np.array(img_list), np.array(label_list)
    factor = sfreq // target_sfreq
    return perception[:, :, ::factor].astype(np.float32), imagery[:, :, ::factor].astype(np.float32), labels, class_map


def load_subject_task_data(dataset_dir, subject, task, apply_ica=True):
    """Load all sessions combined for one subject and one task."""
    dataset_dir = Path(dataset_dir)
    all_perc, all_img, all_labels, class_map = [], [], [], None
    
    for ses in ['ses-01', 'ses-02']:
        bdf_path = dataset_dir / subject / ses / 'eeg' / f'{subject}_{ses}_task-{task}_eeg.bdf'
        events_path = dataset_dir / subject / ses / 'eeg' / f'{subject}_{ses}_task-{task}_events.tsv'
        if not bdf_path.exists():
            continue
        raw, _ = load_and_preprocess_cached(bdf_path, apply_ica=apply_ica)
        events_df = pd.read_csv(events_path, sep='\t')
        perc, img, labels, cm = extract_epochs_from_block(raw, events_df, sfreq=int(raw.info['sfreq']))
        all_perc.append(perc); all_img.append(img); all_labels.append(labels)
        if class_map is None: class_map = cm
    
    if not all_perc:
        return None, None, None, None
    return np.concatenate(all_perc), np.concatenate(all_img), np.concatenate(all_labels), class_map


def load_subject_task_by_session(dataset_dir, subject, task, apply_ica=True):
    """Load each session separately."""
    dataset_dir = Path(dataset_dir)
    sessions = {}
    for ses in ['ses-01', 'ses-02']:
        bdf_path = dataset_dir / subject / ses / 'eeg' / f'{subject}_{ses}_task-{task}_eeg.bdf'
        events_path = dataset_dir / subject / ses / 'eeg' / f'{subject}_{ses}_task-{task}_events.tsv'
        if not bdf_path.exists():
            continue
        raw, _ = load_and_preprocess_cached(bdf_path, apply_ica=apply_ica)
        events_df = pd.read_csv(events_path, sep='\t')
        perc, img, labels, cm = extract_epochs_from_block(raw, events_df, sfreq=int(raw.info['sfreq']))
        sessions[ses] = (perc, img, labels, cm)
    return sessions


In [4]:
# ============================================================
# Model Architecture (v2a: configurable F1, D)
# Primary: EEGNet-8,2 | Comparison: EEGNet-4,2
# ============================================================

class ChannelAttention(nn.Module):
    def __init__(self, n_channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(n_channels, max(n_channels // reduction, 4)), nn.ReLU(),
            nn.Linear(max(n_channels // reduction, 4), n_channels), nn.Sigmoid())
    def forward(self, x):
        w = self.fc(x.mean(dim=-1))
        return x * w.unsqueeze(-1)


class EEGEncoder(nn.Module):
    """EEGNet-inspired encoder. F1=temporal filters, D=spatial per temporal, F2=F1*D."""
    def __init__(self, n_channels=32, F1=8, D=2, temporal_kernel_size=64, pool_size=4, dropout=0.4):
        super().__init__()
        F2 = F1 * D
        self.F1, self.D, self.F2 = F1, D, F2
        
        self.temporal_conv = nn.Sequential(
            nn.Conv1d(n_channels, F1, kernel_size=temporal_kernel_size,
                      padding=temporal_kernel_size // 2, bias=False),
            nn.BatchNorm1d(F1))
        self.spatial_conv = nn.Sequential(
            nn.Conv1d(F1, F2, kernel_size=1, groups=F1, bias=False),
            nn.BatchNorm1d(F2), nn.ELU(), nn.AvgPool1d(pool_size), nn.Dropout(dropout))
        self.channel_attention = ChannelAttention(F2)
        self.sep_conv1 = nn.Sequential(
            nn.Conv1d(F2, F2, kernel_size=16, padding=8, groups=F2, bias=False),
            nn.Conv1d(F2, F2, kernel_size=1, bias=False),
            nn.BatchNorm1d(F2), nn.ELU(), nn.AvgPool1d(pool_size), nn.Dropout(dropout))
        self.sep_conv2 = nn.Sequential(
            nn.Conv1d(F2, F2, kernel_size=8, padding=4, groups=F2, bias=False),
            nn.Conv1d(F2, F2, kernel_size=1, bias=False),
            nn.BatchNorm1d(F2), nn.ELU(), nn.AdaptiveAvgPool1d(1), nn.Dropout(dropout))
        self.modality_emb = nn.Embedding(2, 16)
    
    @property
    def output_dim(self):
        return self.F2 + 16
    
    def forward(self, x, modality):
        x = self.temporal_conv(x)
        x = self.spatial_conv(x)
        x = self.channel_attention(x)
        x = self.sep_conv1(x)
        x = self.sep_conv2(x).squeeze(-1)
        return torch.cat([x, self.modality_emb(modality)], dim=1)


class ProjectionHead(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, output_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, output_dim))
    def forward(self, x):
        return F.normalize(self.net(x), dim=1)


class ClassificationHead(nn.Module):
    def __init__(self, input_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, n_classes))
    def forward(self, x):
        return self.net(x)


class FixedImageEncoder(nn.Module):
    """Fixed class prototypes as simplex vertices on the unit hypersphere."""
    def __init__(self, n_classes=3, embedding_dim=64):
        super().__init__()
        prototypes = torch.zeros(n_classes, embedding_dim)
        if n_classes == 3:
            prototypes[0, 0] = 1.0
            prototypes[1, 0] = -0.5; prototypes[1, 1] = 0.8660254
            prototypes[2, 0] = -0.5; prototypes[2, 1] = -0.8660254
        else:
            torch.manual_seed(42)
            prototypes = torch.randn(n_classes, embedding_dim)
            for _ in range(2000):
                for i in range(n_classes):
                    for j in range(n_classes):
                        if i != j:
                            diff = prototypes[i] - prototypes[j]
                            prototypes[i] += 0.01 * diff / (diff.norm() + 1e-8)
                    prototypes[i] = F.normalize(prototypes[i], dim=0)
        self.register_buffer('prototypes', F.normalize(prototypes, dim=1))
    def forward(self, labels):
        return self.prototypes[labels]


class TriContrastiveModel(nn.Module):
    """Tri-contrastive: perception + imagery + fixed prototypes."""
    def __init__(self, n_channels=32, n_classes=3, embedding_dim=64,
                 eeg_hidden_dim=128, F1=8, D=2):
        super().__init__()
        self.n_classes = n_classes
        self.image_encoder = FixedImageEncoder(n_classes, embedding_dim)
        self.eeg_encoder = EEGEncoder(n_channels=n_channels, F1=F1, D=D)
        feat_dim = self.eeg_encoder.output_dim
        self.eeg_projection = ProjectionHead(feat_dim, eeg_hidden_dim, embedding_dim)
        self.classifier = ClassificationHead(feat_dim, n_classes)

    def forward(self, perception_eeg, imagery_eeg, labels):
        B, dev = labels.shape[0], labels.device
        image_emb = self.image_encoder(labels)
        perc_feat = self.eeg_encoder(perception_eeg, torch.zeros(B, dtype=torch.long, device=dev))
        perc_emb, perc_logits = self.eeg_projection(perc_feat), self.classifier(perc_feat)
        img_feat = self.eeg_encoder(imagery_eeg, torch.ones(B, dtype=torch.long, device=dev))
        img_emb, img_logits = self.eeg_projection(img_feat), self.classifier(img_feat)
        return image_emb, perc_emb, img_emb, perc_logits, img_logits


class DenseCEModel(nn.Module):
    """Pure supervised baseline. Same encoder, imagery-only CE. NO perception/contrastive/prototypes."""
    def __init__(self, n_channels=32, n_classes=3, F1=8, D=2):
        super().__init__()
        self.eeg_encoder = EEGEncoder(n_channels=n_channels, F1=F1, D=D)
        self.classifier = ClassificationHead(self.eeg_encoder.output_dim, n_classes)
    def forward(self, imagery_eeg):
        B, dev = imagery_eeg.shape[0], imagery_eeg.device
        feat = self.eeg_encoder(imagery_eeg, torch.ones(B, dtype=torch.long, device=dev))
        return self.classifier(feat)


# Parameter count comparison
for name, f1, d in [('EEGNet-4,2', 4, 2), ('EEGNet-8,2', 8, 2), ('v1 (32,2)', 32, 2)]:
    m = TriContrastiveModel(n_channels=32, n_classes=3, F1=f1, D=d)
    print(f'  {name}: {sum(p.numel() for p in m.parameters()):,} params')


  EEGNet-4,2: 38,959 params
  EEGNet-8,2: 49,399 params
  v1 (32,2): 118,963 params


In [5]:
# ============================================================
# Loss, Dataset, CV, and Training Helpers
# ============================================================

class TriContrastiveLoss(nn.Module):
    """Joint contrastive + classification loss for tri-modal alignment."""
    def __init__(self, temperature=0.1, w_image_perc=1.0, w_image_img=1.0,
                 w_perc_img=0.2, lambda_within=0.5, lambda_align=1.0, lambda_classify=2.0):
        super().__init__()
        self.temperature = temperature
        self.w_image_perc, self.w_image_img, self.w_perc_img = w_image_perc, w_image_img, w_perc_img
        self.lambda_within, self.lambda_align, self.lambda_classify = lambda_within, lambda_align, lambda_classify
        self.ce = nn.CrossEntropyLoss()

    def _nt_xent(self, z_a, z_b, labels):
        sim = torch.mm(z_a, z_b.t()) / self.temperature
        pos_mask = (labels.unsqueeze(0) == labels.unsqueeze(1)).float()
        pos_mask.fill_diagonal_(0)
        log_softmax = sim - torch.logsumexp(sim, dim=1, keepdim=True)
        pos_count = pos_mask.sum(dim=1).clamp(min=1)
        return (-(pos_mask * log_softmax).sum(dim=1) / pos_count).mean()

    def _align_loss(self, eeg_emb, proto_emb):
        return 1.0 - (eeg_emb * proto_emb).sum(dim=1).mean()

    def forward(self, image_emb, perc_emb, img_emb, labels, perc_logits, img_logits):
        L_within = 0.5 * (self._nt_xent(perc_emb, perc_emb, labels) +
                          self._nt_xent(img_emb, img_emb, labels))
        L_cross = (self._nt_xent(image_emb, perc_emb, labels) * self.w_image_perc +
                   self._nt_xent(image_emb, img_emb, labels) * self.w_image_img +
                   self._nt_xent(perc_emb, img_emb, labels) * self.w_perc_img)
        L_align = 0.5 * (self._align_loss(perc_emb, image_emb) +
                         self._align_loss(img_emb, image_emb))
        L_cls = 0.5 * (self.ce(perc_logits, labels) + self.ce(img_logits, labels))
        total = self.lambda_within * L_within + L_cross + self.lambda_align * L_align + self.lambda_classify * L_cls
        return total, {'total': total.item()}


# -- Ablation mask --
def get_ablation_mask(channel_names, mode='none'):
    mask = np.ones(len(channel_names), dtype=bool)
    if mode == 'frontal':
        for ch in FRONTAL_ABLATION_CHANNELS:
            if ch in channel_names:
                mask[channel_names.index(ch)] = False
    elif mode == 'posterior_only':
        mask[:] = False
        for ch in POSTERIOR_RETAIN_CHANNELS:
            if ch in channel_names:
                mask[channel_names.index(ch)] = True
    return mask


# -- Datasets --
class TriContrastiveDataset(Dataset):
    """Dataset for tri-contrastive model (perception + imagery pairs)."""
    def __init__(self, perception, imagery, labels, crop_length=625, augment=True,
                 channel_dropout_prob=0.1, noise_scale=0.1, ablation_mode='none', scale_uv=True):
        if scale_uv:
            perception, imagery = perception * 1e6, imagery * 1e6
        self.perception, self.imagery, self.labels = perception, imagery, labels
        self.crop_length, self.augment = crop_length, augment
        self.channel_dropout_prob, self.noise_scale = channel_dropout_prob, noise_scale
        self.channel_stds = perception.std(axis=(0, 2))
        self.ablation_mask = get_ablation_mask(CHANNEL_NAMES_32, ablation_mode)
    
    def __len__(self): return len(self.labels)
    
    def __getitem__(self, idx):
        perc, img, label = self.perception[idx].copy(), self.imagery[idx].copy(), self.labels[idx]
        if self.augment:
            perc, img = self._aug(perc), self._aug(img)
        else:
            s = (perc.shape[1] - self.crop_length) // 2
            perc, img = perc[:, s:s+self.crop_length], img[:, s:s+self.crop_length]
        perc[~self.ablation_mask] = 0.0; img[~self.ablation_mask] = 0.0
        return torch.from_numpy(perc), torch.from_numpy(img), torch.tensor(label, dtype=torch.long)
    
    def _aug(self, epoch):
        n_ch, n_t = epoch.shape
        s = np.random.randint(0, n_t - self.crop_length + 1)
        epoch = epoch[:, s:s+self.crop_length]
        epoch = epoch * (np.random.random(n_ch) > self.channel_dropout_prob)[:, None].astype(np.float32)
        return epoch + np.random.randn(*epoch.shape).astype(np.float32) * self.channel_stds[:, None] * self.noise_scale


class ImageryOnlyDataset(Dataset):
    """Dataset for Dense CE baseline - imagery epochs only."""
    def __init__(self, imagery, labels, crop_length=625, augment=True,
                 channel_dropout_prob=0.1, noise_scale=0.1, ablation_mode='none', scale_uv=True):
        if scale_uv: imagery = imagery * 1e6
        self.imagery, self.labels = imagery, labels
        self.crop_length, self.augment = crop_length, augment
        self.channel_dropout_prob, self.noise_scale = channel_dropout_prob, noise_scale
        self.channel_stds = imagery.std(axis=(0, 2))
        self.ablation_mask = get_ablation_mask(CHANNEL_NAMES_32, ablation_mode)
    
    def __len__(self): return len(self.labels)
    
    def __getitem__(self, idx):
        img, label = self.imagery[idx].copy(), self.labels[idx]
        if self.augment:
            n_ch, n_t = img.shape
            s = np.random.randint(0, n_t - self.crop_length + 1)
            img = img[:, s:s+self.crop_length]
            img = img * (np.random.random(n_ch) > self.channel_dropout_prob)[:, None].astype(np.float32)
            img = img + np.random.randn(*img.shape).astype(np.float32) * self.channel_stds[:, None] * self.noise_scale
        else:
            s = (img.shape[1] - self.crop_length) // 2
            img = img[:, s:s+self.crop_length]
        img[~self.ablation_mask] = 0.0
        return torch.from_numpy(img), torch.tensor(label, dtype=torch.long)


# -- Purged Temporal Block CV --
def get_temporal_block_splits(labels, n_splits=5):
    n = len(labels)
    blocks = np.array_split(np.arange(n), n_splits)
    splits = []
    for i in range(n_splits):
        test_idx = blocks[i]
        train_idx = np.concatenate([blocks[j] for j in range(n_splits) if j != i])
        test_set = set(test_idx)
        boundary = set()
        for t in test_idx:
            if t - 1 >= 0 and t - 1 not in test_set: boundary.add(t - 1)
            if t + 1 < n and t + 1 not in test_set: boundary.add(t + 1)
        train_idx = np.array([t for t in train_idx if t not in boundary])
        rng = np.random.RandomState(42); rng.shuffle(train_idx)
        splits.append((train_idx, test_idx))
    return splits


class TemporalBlockCV:
    def __init__(self, labels, n_splits=5):
        self.splits = get_temporal_block_splits(labels, n_splits)
        self.n_splits = n_splits
    def split(self, X, y=None, groups=None):
        for tr, te in self.splits: yield tr, te
    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits


def sig_str(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'n.s.'

def cohens_d(a, b=None):
    diff = a - b if b is not None else a
    return diff.mean() / (diff.std() + 1e-10)


# -- Training functions --
def _train_loop(model, criterion_fn, train_loader, test_loader, optimizer, scheduler,
                n_epochs, patience_limit, device):
    best_val, patience, best_state = float('inf'), 0, None
    for epoch in range(n_epochs):
        model.train()
        for batch in train_loader:
            batch = [b.to(device) for b in batch]
            optimizer.zero_grad()
            loss = criterion_fn(model, batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        
        model.eval()
        vl, n = 0, 0
        with torch.no_grad():
            for batch in test_loader:
                batch = [b.to(device) for b in batch]
                vl += criterion_fn(model, batch).item(); n += 1
        vl /= max(n, 1)
        if vl < best_val:
            best_val, patience = vl, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience += 1
        if patience >= patience_limit: break
    
    model.load_state_dict(best_state)
    return model.to(device)


def train_tri_model(train_loader, test_loader, hp, n_classes, device,
                    F1=8, D=2, n_epochs=150, patience_limit=25):
    model = TriContrastiveModel(n_channels=32, n_classes=n_classes,
        embedding_dim=hp.get('emb_dim', 64), eeg_hidden_dim=hp.get('hidden_dim', 128),
        F1=F1, D=D).to(device)
    criterion = TriContrastiveLoss(
        temperature=hp.get('temperature', 0.1), w_image_perc=hp.get('w_img_perc', 1.0),
        w_image_img=hp.get('w_img_img', 1.0), w_perc_img=hp.get('w_perc_img', 0.2),
        lambda_within=hp.get('lam_within', 0.5), lambda_align=hp.get('lam_align', 1.0),
        lambda_classify=hp.get('lam_classify', 2.0))
    
    def crit_fn(mdl, batch):
        p_b, i_b, l_b = batch
        ie, pe, ime, pl, il = mdl(p_b, i_b, l_b)
        loss, _ = criterion(ie, pe, ime, l_b, pl, il)
        return loss
    
    opt = torch.optim.AdamW(model.parameters(), lr=hp.get('lr', 5e-4), weight_decay=hp.get('wd', 1e-2))
    warmup = LinearLR(opt, start_factor=0.01, total_iters=10)
    cosine = CosineAnnealingLR(opt, T_max=n_epochs - 10, eta_min=1e-6)
    sched = SequentialLR(opt, [warmup, cosine], milestones=[10])
    return _train_loop(model, crit_fn, train_loader, test_loader, opt, sched, n_epochs, patience_limit, device)


def train_dense_ce_model(train_loader, test_loader, hp, n_classes, device,
                         F1=8, D=2, n_epochs=150, patience_limit=25):
    """Train imagery-only model with pure CE. NO perception. NO contrastive. NO prototypes."""
    model = DenseCEModel(n_channels=32, n_classes=n_classes, F1=F1, D=D).to(device)
    ce = nn.CrossEntropyLoss()
    
    def crit_fn(mdl, batch):
        img_b, lab_b = batch
        return ce(mdl(img_b), lab_b)
    
    opt = torch.optim.AdamW(model.parameters(), lr=hp.get('lr', 5e-4), weight_decay=hp.get('wd', 1e-2))
    warmup = LinearLR(opt, start_factor=0.01, total_iters=10)
    cosine = CosineAnnealingLR(opt, T_max=n_epochs - 10, eta_min=1e-6)
    sched = SequentialLR(opt, [warmup, cosine], milestones=[10])
    return _train_loop(model, crit_fn, train_loader, test_loader, opt, sched, n_epochs, patience_limit, device)


# -- Probing --
def probe_tri_model(model, train_loader, test_loader, device):
    model.eval()
    def get_embs(loader):
        ei, ep, lab = [], [], []
        with torch.no_grad():
            for p_b, i_b, l_b in loader:
                p_b, i_b, l_b = p_b.to(device), i_b.to(device), l_b.to(device)
                _, pe, ime, _, _ = model(p_b, i_b, l_b)
                ei.append(ime.cpu().numpy()); ep.append(pe.cpu().numpy()); lab.append(l_b.cpu().numpy())
        return np.concatenate(ei), np.concatenate(ep), np.concatenate(lab)
    
    tr_i, tr_p, tr_l = get_embs(train_loader)
    te_i, te_p, te_l = get_embs(test_loader)
    lr_i = LogisticRegression(max_iter=1000, random_state=42).fit(tr_i, tr_l)
    preds = lr_i.predict(te_i)
    lr_p = LogisticRegression(max_iter=1000, random_state=42).fit(tr_p, tr_l)
    return accuracy_score(te_l, preds), accuracy_score(te_l, lr_p.predict(te_p)), preds, te_l


def probe_dense_ce(model, test_loader, device):
    model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for img_b, lab_b in test_loader:
            logits = model(img_b.to(device))
            all_p.append(logits.argmax(1).cpu().numpy()); all_l.append(lab_b.numpy())
    all_p, all_l = np.concatenate(all_p), np.concatenate(all_l)
    return accuracy_score(all_l, all_p), all_p, all_l


# -- Loader factories --
def make_tri_loaders(p_tr, i_tr, l_tr, p_te, i_te, l_te, hp, ablation_mode='none'):
    trd = TriContrastiveDataset(p_tr, i_tr, l_tr, CROP_LENGTH, True,
        hp.get('ch_dropout', 0.1), hp.get('noise_scale', 0.1), ablation_mode)
    ted = TriContrastiveDataset(p_te, i_te, l_te, CROP_LENGTH, False, ablation_mode=ablation_mode)
    return (DataLoader(trd, batch_size=hp.get('batch_size', 32), shuffle=True, drop_last=True, num_workers=0),
            DataLoader(ted, batch_size=64, shuffle=False, num_workers=0))


def make_img_loaders(i_tr, l_tr, i_te, l_te, hp, ablation_mode='none'):
    trd = ImageryOnlyDataset(i_tr, l_tr, CROP_LENGTH, True,
        hp.get('ch_dropout', 0.1), hp.get('noise_scale', 0.1), ablation_mode)
    ted = ImageryOnlyDataset(i_te, l_te, CROP_LENGTH, False, ablation_mode=ablation_mode)
    return (DataLoader(trd, batch_size=hp.get('batch_size', 32), shuffle=True, drop_last=True, num_workers=0),
            DataLoader(ted, batch_size=64, shuffle=False, num_workers=0))


# -- Full CV runners --
def run_tri_cv(perc, img, labels, hp, n_cls, device, F1=8, D=2,
               ablation_mode='none', return_preds=False, n_splits=5):
    splits = get_temporal_block_splits(labels, n_splits)
    img_accs, perc_accs, all_preds, all_labels = [], [], [], []
    for tr, te in splits:
        trl, tel = make_tri_loaders(perc[tr], img[tr], labels[tr], perc[te], img[te], labels[te], hp, ablation_mode)
        model = train_tri_model(trl, tel, hp, n_cls, device, F1=F1, D=D)
        ia, pa, preds, labs = probe_tri_model(model, trl, tel, device)
        img_accs.append(ia); perc_accs.append(pa)
        if return_preds: all_preds.append(preds); all_labels.append(labs)
    result = {'img': np.mean(img_accs), 'perc': np.mean(perc_accs),
              'img_folds': img_accs, 'perc_folds': perc_accs}
    if return_preds:
        result['preds'] = np.concatenate(all_preds); result['labels'] = np.concatenate(all_labels)
    return result


def run_dense_ce_cv(img, labels, hp, n_cls, device, F1=8, D=2, ablation_mode='none', n_splits=5):
    splits = get_temporal_block_splits(labels, n_splits)
    img_accs = []
    for tr, te in splits:
        trl, tel = make_img_loaders(img[tr], labels[tr], img[te], labels[te], hp, ablation_mode)
        model = train_dense_ce_model(trl, tel, hp, n_cls, device, F1=F1, D=D)
        acc, _, _ = probe_dense_ce(model, tel, device)
        img_accs.append(acc)
    return {'img': np.mean(img_accs), 'img_folds': img_accs}

print("All helpers defined.")


All helpers defined.


In [6]:
# ============================================================
# PART A: FBCSP Baselines (all 22 subjects)
# 9 non-overlapping 4Hz sub-bands, MI feature selection, LDA/SVM
# ============================================================

def bandpass_filter_epochs(epochs, low, high, sfreq=250, order=4):
    sos = butter(order, [low, high], btype='band', fs=sfreq, output='sos')
    return sosfilt(sos, epochs, axis=-1).astype(np.float64)


def fbcsp_features(X, labels, X_test=None, n_components=4, sfreq=250):
    """9-band FBCSP feature extraction."""
    bands = [(4*i, 4*i+4) for i in range(1, 10)]  # (4,8)...(36,40)
    train_feats, test_feats = [], []
    
    for low, high in bands:
        X_band = bandpass_filter_epochs(X, low, high, sfreq)
        n_comp = min(n_components, X_band.shape[1] - 1, len(np.unique(labels)) * 2)
        csp = CSP(n_components=n_comp, reg='ledoit_wolf', log=True)
        train_feats.append(csp.fit_transform(X_band, labels))
        if X_test is not None:
            test_feats.append(csp.transform(bandpass_filter_epochs(X_test, low, high, sfreq)))
    
    if X_test is not None:
        return np.hstack(train_feats), np.hstack(test_feats)
    return np.hstack(train_feats)


K_BEST = 18  # MI feature selection: top 18 of ~36 features

print('=' * 70)
print('PART A: FBCSP BASELINES (9 bands, MI feature selection)')
print('=' * 70)

fbcsp_results = []
for task in ['AVI', 'FVI', 'OVI']:
    n_cls = TASKS[task]['n_classes']
    n_csp = min(4, n_cls * 2)
    print(f'\n--- {task} ({n_cls} classes) ---')
    
    for sub in subjects_all:
        perc, img, labels, _ = load_subject_task_data(DATASET_DIR, sub, task)
        if img is None or len(np.unique(labels)) < n_cls:
            continue
        
        cv = TemporalBlockCV(labels, n_splits=N_FOLDS)
        row = {'subject': sub, 'task': task}
        
        for mod, X in [('imagery', img), ('perception', perc)]:
            fold_lda, fold_svm = [], []
            for train_idx, test_idx in cv.split(X):
                feat_tr, feat_te = fbcsp_features(X[train_idx], labels[train_idx], X[test_idx], n_csp)
                
                # MI feature selection
                k = min(K_BEST, feat_tr.shape[1])
                sel = SelectKBest(mutual_info_classif, k=k)
                ft_sel, fte_sel = sel.fit_transform(feat_tr, labels[train_idx]), sel.transform(feat_te)
                
                lda = LinearDiscriminantAnalysis().fit(ft_sel, labels[train_idx])
                fold_lda.append(accuracy_score(labels[test_idx], lda.predict(fte_sel)))
                
                sc = StandardScaler()
                svm = SVC(kernel='rbf', C=1.0, gamma='scale')
                svm.fit(sc.fit_transform(ft_sel), labels[train_idx])
                fold_svm.append(accuracy_score(labels[test_idx], svm.predict(sc.transform(fte_sel))))
            
            row[f'lda_{mod}'] = np.mean(fold_lda)
            row[f'svm_{mod}'] = np.mean(fold_svm)
        
        fbcsp_results.append(row)
        print(f'  {sub}: LDA(I:{row["lda_imagery"]:.3f}) SVM(I:{row["svm_imagery"]:.3f})')

fbcsp_df = pd.DataFrame(fbcsp_results)
fbcsp_df.to_csv(OUTPUT_DIR / 'fbcsp_baselines_v2a.csv', index=False)
print(f'\nSaved {len(fbcsp_df)} FBCSP results')


PART A: FBCSP BASELINES (9 bands, MI feature selection)

--- AVI (3 classes) ---
  ICA cached: sub-01_ses-01_task-AVI_eeg.bdf (removed 4)
  ICA cached: sub-01_ses-02_task-AVI_eeg.bdf (removed 3)
  sub-01: LDA(I:0.342) SVM(I:0.338)
  ICA cached: sub-02_ses-01_task-AVI_eeg.bdf (removed 4)
  ICA cached: sub-02_ses-02_task-AVI_eeg.bdf (removed 5)
  sub-02: LDA(I:0.317) SVM(I:0.325)
  ICA cached: sub-03_ses-01_task-AVI_eeg.bdf (removed 3)
  ICA cached: sub-03_ses-02_task-AVI_eeg.bdf (removed 3)
  sub-03: LDA(I:0.354) SVM(I:0.321)
  ICA cached: sub-04_ses-01_task-AVI_eeg.bdf (removed 2)
  ICA cached: sub-04_ses-02_task-AVI_eeg.bdf (removed 2)
  sub-04: LDA(I:0.250) SVM(I:0.271)
  ICA cached: sub-05_ses-01_task-AVI_eeg.bdf (removed 4)
  ICA cached: sub-05_ses-02_task-AVI_eeg.bdf (removed 4)
  sub-05: LDA(I:0.317) SVM(I:0.283)
  ICA cached: sub-06_ses-01_task-AVI_eeg.bdf (removed 5)
  ICA cached: sub-06_ses-02_task-AVI_eeg.bdf (removed 7)
  sub-06: LDA(I:0.371) SVM(I:0.346)
  ICA cached: sub-0

In [7]:
# ============================================================
# OPTUNA SUBJECT SELECTION + HARDCODED EXCLUSION
# Rank by FBCSP/LDA, pick 25th/50th/75th percentile, REMOVE from eval
# ============================================================

sub_mean_fbcsp = (
    fbcsp_df.groupby('subject')['lda_imagery'].mean()
    .sort_values().reset_index()
)
sub_mean_fbcsp['rank'] = range(1, len(sub_mean_fbcsp) + 1)
sub_mean_fbcsp['pctl'] = sub_mean_fbcsp['rank'] / len(sub_mean_fbcsp)

print('FBCSP/LDA imagery accuracy ranking:')
print(sub_mean_fbcsp.to_string(index=False))

n = len(sub_mean_fbcsp)
SEARCH_SUBJECTS = [
    sub_mean_fbcsp.iloc[int(n * 0.25)]['subject'],
    sub_mean_fbcsp.iloc[int(n * 0.50)]['subject'],
    sub_mean_fbcsp.iloc[int(n * 0.75)]['subject'],
]

print(f'\nSelected Optuna search subjects (25th/50th/75th percentile):')
for s in SEARCH_SUBJECTS:
    row = sub_mean_fbcsp[sub_mean_fbcsp['subject'] == s].iloc[0]
    print(f'  {s}: FBCSP/LDA mean = {row["lda_imagery"]:.3f} (rank {int(row["rank"])})')

# ═══════════════════════════════════════════════════════════════
# CRITICAL: Physically remove search subjects from evaluation.
# All downstream loops use subjects_eval ONLY.
# ═══════════════════════════════════════════════════════════════
subjects_eval = [s for s in subjects_all if s not in SEARCH_SUBJECTS]

assert len(subjects_eval) == len(subjects_all) - len(SEARCH_SUBJECTS)
for s in SEARCH_SUBJECTS:
    assert s not in subjects_eval, f'{s} still in eval list!'

print(f'\nEvaluation subjects ({len(subjects_eval)}): {subjects_eval}')
print(f'EXCLUDED from all evaluation: {SEARCH_SUBJECTS}')
print('Subject isolation VERIFIED')


FBCSP/LDA imagery accuracy ranking:
subject  lda_imagery  rank     pctl
 sub-14     0.276042     1 0.045455
 sub-11     0.277431     2 0.090909
 sub-04     0.278819     3 0.136364
 sub-02     0.285417     4 0.181818
 sub-18     0.288542     5 0.227273
 sub-01     0.304514     6 0.272727
 sub-19     0.306944     7 0.318182
 sub-15     0.308681     8 0.363636
 sub-13     0.309722     9 0.409091
 sub-05     0.310069    10 0.454545
 sub-08     0.310764    11 0.500000
 sub-10     0.312500    12 0.545455
 sub-16     0.315972    13 0.590909
 sub-20     0.316667    14 0.636364
 sub-06     0.317361    15 0.681818
 sub-12     0.328472    16 0.727273
 sub-09     0.331944    17 0.772727
 sub-03     0.342014    18 0.818182
 sub-17     0.345139    19 0.863636
 sub-07     0.345833    20 0.909091
 sub-22     0.355903    21 0.954545
 sub-21     0.439931    22 1.000000

Selected Optuna search subjects (25th/50th/75th percentile):
  sub-01: FBCSP/LDA mean = 0.305 (rank 6)
  sub-10: FBCSP/LDA mean = 0.312

In [8]:
# ============================================================
# OPTUNA HP SEARCH (on search subjects only, EEGNet-8,2)
# ============================================================
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

hp_path = OUTPUT_DIR / 'best_hparams_v2a.json'

print(f'Preloading search data for {SEARCH_SUBJECTS} on {SEARCH_TASK}...')
search_data = {}
for sub in SEARCH_SUBJECTS:
    perc, img, labels, _ = load_subject_task_data(DATASET_DIR, sub, SEARCH_TASK)
    if perc is not None:
        search_data[sub] = (perc, img, labels)
        print(f'  {sub}: {len(labels)} trials')

search_splits = {sub: get_temporal_block_splits(labels, SEARCH_N_FOLDS)
                 for sub, (_, _, labels) in search_data.items()}


def objective(trial):
    hp = {
        'lr': trial.suggest_float('lr', 1e-4, 2e-3, log=True),
        'wd': trial.suggest_float('wd', 1e-3, 5e-2, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [24, 32, 48]),
        'emb_dim': trial.suggest_categorical('emb_dim', [48, 64, 96]),
        'hidden_dim': trial.suggest_categorical('hidden_dim', [96, 128, 192]),
        'temperature': trial.suggest_float('temperature', 0.05, 0.3, log=True),
        'w_img_img': trial.suggest_float('w_img_img', 0.5, 2.0),
        'w_img_perc': 1.0,
        'w_perc_img': trial.suggest_float('w_perc_img', 0.05, 0.5),
        'lam_within': trial.suggest_float('lam_within', 0.1, 1.5),
        'lam_align': trial.suggest_float('lam_align', 0.3, 3.0),
        'lam_classify': trial.suggest_float('lam_classify', 0.5, 5.0),
        'ch_dropout': trial.suggest_float('ch_dropout', 0.05, 0.2),
        'noise_scale': trial.suggest_float('noise_scale', 0.05, 0.2),
    }
    accs = []
    for sub, (perc, img, labels) in search_data.items():
        for tr, te in search_splits[sub]:
            trl, tel = make_tri_loaders(perc[tr], img[tr], labels[tr], perc[te], img[te], labels[te], hp)
            model = train_tri_model(trl, tel, hp, TASKS[SEARCH_TASK]['n_classes'], device,
                                    F1=8, D=2, n_epochs=SEARCH_EPOCHS, patience_limit=SEARCH_PATIENCE)
            ia, _, _, _ = probe_tri_model(model, trl, tel, device)
            accs.append(ia)
            trial.report(np.mean(accs), len(accs))
            if trial.should_prune(): raise optuna.TrialPruned()
    return np.mean(accs)


if hp_path.exists():
    with open(hp_path) as f: bp = json.load(f)
    print(f'Loaded cached hparams from {hp_path}')
else:
    study = optuna.create_study(direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=3))
    print(f'Running {N_OPTUNA_TRIALS} Optuna trials...')
    t0 = time.time()
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    print(f'Search done in {(time.time()-t0)/60:.1f} min, best acc: {study.best_value:.4f}')
    bp = study.best_params
    bp['w_img_perc'] = 1.0
    with open(hp_path, 'w') as f: json.dump(bp, f, indent=2)
    study.trials_dataframe().to_csv(OUTPUT_DIR / 'optuna_study_v2a.csv', index=False)

HP = {**bp, 'w_img_perc': 1.0}
print(f'\nHyperparameters: {HP}')


Preloading search data for ['sub-01', 'sub-10', 'sub-09'] on FVI...
  sub-01: 240 trials
  sub-10: 120 trials
  sub-09: 120 trials
Running 50 Optuna trials...


Best trial: 0. Best value: 0.345833: 100%|██████████| 50/50 [11:40<00:00, 14.00s/it]

Search done in 11.7 min, best acc: 0.3458

Hyperparameters: {'lr': 0.0003071057367777374, 'wd': 0.04123206532618727, 'batch_size': 24, 'emb_dim': 96, 'hidden_dim': 128, 'temperature': 0.2842539893932711, 'w_img_img': 1.7486639612006325, 'w_perc_img': 0.14555259980522428, 'lam_within': 0.35455495408994087, 'lam_align': 0.7951921766042713, 'lam_classify': 1.8690900933179198, 'ch_dropout': 0.1287134647448357, 'noise_scale': 0.11479175279631737, 'w_img_perc': 1.0}


In [9]:
# ============================================================
# PART B: TRI-CONTRASTIVE EVALUATION (8,2 and 4,2)
# Evaluated on subjects_eval ONLY (search subjects excluded)
# ============================================================
print('=' * 70)
print(f'PART B: TRI-CONTRASTIVE on {len(subjects_eval)} eval subjects')
print('=' * 70)

tri_results = []
for F1_eval, D_eval, tag in [(8, 2, '8_2'), (4, 2, '4_2')]:
    print(f'\n=== Architecture: EEGNet-{F1_eval},{D_eval} ===')
    for task in ['AVI', 'FVI', 'OVI']:
        n_cls = TASKS[task]['n_classes']
        print(f'\n--- {task} ---')
        t0 = time.time()
        for i, sub in enumerate(subjects_eval):
            perc, img, labels, _ = load_subject_task_data(DATASET_DIR, sub, task)
            if perc is None or len(np.unique(labels)) < n_cls:
                print(f'  [{i+1}/{len(subjects_eval)}] {sub}: SKIP'); continue
            return_p = (task == 'FVI' and tag == '8_2')
            res = run_tri_cv(perc, img, labels, HP, n_cls, device, F1=F1_eval, D=D_eval, return_preds=return_p)
            row = {'subject': sub, 'task': task, 'arch': tag,
                   'tri_img': res['img'], 'tri_perc': res['perc'],
                   'tri_img_folds': res['img_folds'], 'tri_perc_folds': res['perc_folds']}
            if return_p: row['_preds'] = res['preds']; row['_labels'] = res['labels']
            tri_results.append(row)
            print(f'  [{i+1}/{len(subjects_eval)}] {sub}: I:{res["img"]:.3f} P:{res["perc"]:.3f}')
        print(f'  [{task} done in {(time.time()-t0)/60:.1f} min]')

tri_df_save = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in tri_results])
tri_df_save.to_csv(OUTPUT_DIR / 'tri_results_v2a.csv', index=False)


PART B: TRI-CONTRASTIVE on 19 eval subjects

=== Architecture: EEGNet-8,2 ===

--- AVI ---
  [1/19] sub-02: I:0.358 P:0.396
  [2/19] sub-03: I:0.350 P:0.346
  [3/19] sub-04: I:0.329 P:0.354
  [4/19] sub-05: I:0.421 P:0.400
  [5/19] sub-06: I:0.317 P:0.300
  [6/19] sub-07: I:0.417 P:0.383
  [7/19] sub-08: I:0.392 P:0.350
  [8/19] sub-11: I:0.417 P:0.325
  [9/19] sub-12: I:0.362 P:0.363
  [10/19] sub-13: I:0.412 P:0.421
  [11/19] sub-14: I:0.338 P:0.329
  [12/19] sub-15: I:0.362 P:0.371
  [13/19] sub-16: I:0.371 P:0.375
  [14/19] sub-17: I:0.392 P:0.388
  [15/19] sub-18: I:0.329 P:0.321
  [16/19] sub-19: I:0.392 P:0.383
  [17/19] sub-20: I:0.312 P:0.317
  [18/19] sub-21: I:0.371 P:0.329
  [19/19] sub-22: I:0.371 P:0.358
  [AVI done in 11.4 min]

--- FVI ---
  [1/19] sub-02: I:0.346 P:0.375
  [2/19] sub-03: I:0.383 P:0.362
  [3/19] sub-04: I:0.362 P:0.346
  [4/19] sub-05: I:0.363 P:0.446
  [5/19] sub-06: I:0.321 P:0.354
  [6/19] sub-07: I:0.333 P:0.367
  [7/19] sub-08: I:0.337 P:0.367
  [

In [10]:
# ============================================================
# PART C: DENSE CE BASELINE (imagery-only, pure cross-entropy)
# Same EEGEncoder(8,2), NO perception, NO contrastive, NO prototypes
# ============================================================
print('=' * 70)
print(f'PART C: DENSE CE BASELINE on {len(subjects_eval)} eval subjects')
print('=' * 70)

dense_ce_results = []
for task in ['AVI', 'FVI', 'OVI']:
    n_cls = TASKS[task]['n_classes']
    print(f'\n--- {task} ---')
    t0 = time.time()
    for i, sub in enumerate(subjects_eval):
        _, img, labels, _ = load_subject_task_data(DATASET_DIR, sub, task)
        if img is None or len(np.unique(labels)) < n_cls:
            print(f'  [{i+1}/{len(subjects_eval)}] {sub}: SKIP'); continue
        res = run_dense_ce_cv(img, labels, HP, n_cls, device, F1=8, D=2)
        dense_ce_results.append({'subject': sub, 'task': task,
            'dense_ce_img': res['img'], 'dense_ce_img_folds': res['img_folds']})
        print(f'  [{i+1}/{len(subjects_eval)}] {sub}: I:{res["img"]:.3f}')
    print(f'  [{task} done in {(time.time()-t0)/60:.1f} min]')

dense_ce_df = pd.DataFrame(dense_ce_results)
dense_ce_df.to_csv(OUTPUT_DIR / 'dense_ce_results_v2a.csv', index=False)


PART C: DENSE CE BASELINE on 19 eval subjects

--- AVI ---
  [1/19] sub-02: I:0.358
  [2/19] sub-03: I:0.379
  [3/19] sub-04: I:0.325
  [4/19] sub-05: I:0.412
  [5/19] sub-06: I:0.325
  [6/19] sub-07: I:0.350
  [7/19] sub-08: I:0.417
  [8/19] sub-11: I:0.312
  [9/19] sub-12: I:0.404
  [10/19] sub-13: I:0.396
  [11/19] sub-14: I:0.342
  [12/19] sub-15: I:0.321
  [13/19] sub-16: I:0.362
  [14/19] sub-17: I:0.396
  [15/19] sub-18: I:0.408
  [16/19] sub-19: I:0.333
  [17/19] sub-20: I:0.333
  [18/19] sub-21: I:0.358
  [19/19] sub-22: I:0.400
  [AVI done in 4.4 min]

--- FVI ---
  [1/19] sub-02: I:0.392
  [2/19] sub-03: I:0.379
  [3/19] sub-04: I:0.408
  [4/19] sub-05: I:0.429
  [5/19] sub-06: I:0.329
  [6/19] sub-07: I:0.408
  [7/19] sub-08: I:0.363
  [8/19] sub-11: I:0.392
  [9/19] sub-12: I:0.396
  [10/19] sub-13: I:0.392
  [11/19] sub-14: I:0.412
  [12/19] sub-15: I:0.396
  [13/19] sub-16: I:0.375
  [14/19] sub-17: I:0.362
  [15/19] sub-18: I:0.375
  [16/19] sub-19: I:0.346
  [17/19] su

In [11]:
# ============================================================
# PART D: ABLATION STUDIES
# 1. Frontal-7: zero Fp1/Fp2/Fpz/F7/F8/FT7/FT8
# 2. Posterior-only: retain only posterior/central channels
# ============================================================
print('=' * 70)
print('PART D: ABLATION STUDIES')
print('=' * 70)

ablation_results = []
for abl_mode, abl_name in [('frontal', 'frontal_7'), ('posterior_only', 'posterior_only')]:
    retained = [ch for ch, m in zip(CHANNEL_NAMES_32, get_ablation_mask(CHANNEL_NAMES_32, abl_mode)) if m]
    print(f'\n=== {abl_name}: {len(retained)} active channels ===')
    
    for task in ['AVI', 'FVI', 'OVI']:
        n_cls = TASKS[task]['n_classes']
        print(f'\n--- {task} ---')
        t0 = time.time()
        for i, sub in enumerate(subjects_eval):
            perc, img, labels, _ = load_subject_task_data(DATASET_DIR, sub, task)
            if perc is None or len(np.unique(labels)) < n_cls: continue
            res = run_tri_cv(perc, img, labels, HP, n_cls, device, F1=8, D=2, ablation_mode=abl_mode)
            ablation_results.append({'subject': sub, 'task': task, 'ablation': abl_name,
                'abl_img': res['img'], 'abl_perc': res['perc']})
            print(f'  [{i+1}/{len(subjects_eval)}] {sub}: I:{res["img"]:.3f} P:{res["perc"]:.3f}')
        print(f'  [{task} done in {(time.time()-t0)/60:.1f} min]')

ablation_df = pd.DataFrame(ablation_results)
ablation_df.to_csv(OUTPUT_DIR / 'ablation_results_v2a.csv', index=False)


PART D: ABLATION STUDIES

=== frontal_7: 25 active channels ===

--- AVI ---
  [1/19] sub-02: I:0.375 P:0.346
  [2/19] sub-03: I:0.371 P:0.321
  [3/19] sub-04: I:0.317 P:0.375
  [4/19] sub-05: I:0.404 P:0.438
  [5/19] sub-06: I:0.342 P:0.333
  [6/19] sub-07: I:0.388 P:0.412
  [7/19] sub-08: I:0.308 P:0.292
  [8/19] sub-11: I:0.325 P:0.325
  [9/19] sub-12: I:0.408 P:0.392
  [10/19] sub-13: I:0.446 P:0.429
  [11/19] sub-14: I:0.304 P:0.321
  [12/19] sub-15: I:0.367 P:0.392
  [13/19] sub-16: I:0.375 P:0.371
  [14/19] sub-17: I:0.362 P:0.392
  [15/19] sub-18: I:0.338 P:0.354
  [16/19] sub-19: I:0.362 P:0.383
  [17/19] sub-20: I:0.304 P:0.308
  [18/19] sub-21: I:0.346 P:0.325
  [19/19] sub-22: I:0.358 P:0.383
  [AVI done in 10.9 min]

--- FVI ---
  [1/19] sub-02: I:0.375 P:0.379
  [2/19] sub-03: I:0.375 P:0.329
  [3/19] sub-04: I:0.350 P:0.375
  [4/19] sub-05: I:0.396 P:0.467
  [5/19] sub-06: I:0.329 P:0.317
  [6/19] sub-07: I:0.396 P:0.400
  [7/19] sub-08: I:0.329 P:0.354
  [8/19] sub-11: 

In [12]:
# ============================================================
# PART E: CROSS-SESSION GENERALIZATION
# Train ses-01 -> test ses-02 (and reverse). Zero temporal leakage.
# ============================================================
print('=' * 70)
print('PART E: CROSS-SESSION GENERALIZATION')
print('=' * 70)

cross_results = []
for task in ['AVI', 'FVI', 'OVI']:
    n_cls = TASKS[task]['n_classes']
    print(f'\n--- {task} ---')
    for sub in subjects_eval:
        sessions = load_subject_task_by_session(DATASET_DIR, sub, task)
        if 'ses-01' not in sessions or 'ses-02' not in sessions: continue
        p1, i1, l1, _ = sessions['ses-01']
        p2, i2, l2, _ = sessions['ses-02']
        if len(np.unique(l1)) < n_cls or len(np.unique(l2)) < n_cls: continue
        
        trl, tel = make_tri_loaders(p1, i1, l1, p2, i2, l2, HP)
        m = train_tri_model(trl, tel, HP, n_cls, device, F1=8, D=2)
        ia_12, pa_12, _, _ = probe_tri_model(m, trl, tel, device)
        
        trl, tel = make_tri_loaders(p2, i2, l2, p1, i1, l1, HP)
        m = train_tri_model(trl, tel, HP, n_cls, device, F1=8, D=2)
        ia_21, pa_21, _, _ = probe_tri_model(m, trl, tel, device)
        
        cross_results.append({'subject': sub, 'task': task,
            'img_s1_to_s2': ia_12, 'img_s2_to_s1': ia_21,
            'perc_s1_to_s2': pa_12, 'perc_s2_to_s1': pa_21,
            'img_mean': (ia_12 + ia_21) / 2, 'perc_mean': (pa_12 + pa_21) / 2})
        print(f'  {sub}: I(1>2:{ia_12:.3f} 2>1:{ia_21:.3f})')

cross_df = pd.DataFrame(cross_results)
cross_df.to_csv(OUTPUT_DIR / 'cross_session_v2a.csv', index=False)


PART E: CROSS-SESSION GENERALIZATION

--- AVI ---
  sub-02: I(1>2:0.350 2>1:0.308)
  sub-03: I(1>2:0.308 2>1:0.333)
  sub-04: I(1>2:0.375 2>1:0.342)
  sub-05: I(1>2:0.433 2>1:0.367)
  sub-06: I(1>2:0.392 2>1:0.342)
  sub-07: I(1>2:0.400 2>1:0.450)
  sub-11: I(1>2:0.367 2>1:0.408)
  sub-12: I(1>2:0.367 2>1:0.325)
  sub-13: I(1>2:0.392 2>1:0.350)
  sub-14: I(1>2:0.325 2>1:0.308)
  sub-15: I(1>2:0.275 2>1:0.358)
  sub-16: I(1>2:0.367 2>1:0.417)
  sub-17: I(1>2:0.325 2>1:0.342)
  sub-18: I(1>2:0.442 2>1:0.417)
  sub-19: I(1>2:0.425 2>1:0.375)
  sub-20: I(1>2:0.317 2>1:0.308)
  sub-21: I(1>2:0.300 2>1:0.325)
  sub-22: I(1>2:0.400 2>1:0.367)

--- FVI ---
  sub-02: I(1>2:0.325 2>1:0.325)
  sub-03: I(1>2:0.425 2>1:0.417)
  sub-04: I(1>2:0.350 2>1:0.342)
  sub-05: I(1>2:0.408 2>1:0.308)
  sub-06: I(1>2:0.317 2>1:0.358)
  sub-07: I(1>2:0.333 2>1:0.200)
  sub-08: I(1>2:0.350 2>1:0.317)
  sub-11: I(1>2:0.350 2>1:0.308)
  sub-12: I(1>2:0.275 2>1:0.383)
  sub-13: I(1>2:0.383 2>1:0.442)
  sub-14: I(1

In [13]:
# ============================================================
# PART F: CONFUSION MATRICES (FVI, EEGNet-8,2)
# ============================================================
print('=' * 70)
print('PART F: CONFUSION MATRICES')
print('=' * 70)

fvi_preds, fvi_labels = [], []
for r in tri_results:
    if r['task'] == 'FVI' and r['arch'] == '8_2' and '_preds' in r:
        fvi_preds.append(r['_preds']); fvi_labels.append(r['_labels'])

if fvi_preds:
    fvi_p, fvi_l = np.concatenate(fvi_preds), np.concatenate(fvi_labels)
    cm = confusion_matrix(fvi_l, fvi_p)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cn = TASKS['FVI']['classes']
    print(f'\nNormalized confusion matrix:')
    print(f'  {"":12s}', '  '.join(f'{c:>10s}' for c in cn))
    for i, c in enumerate(cn):
        print(f'  {c:12s}', '  '.join(f'{cm_norm[i,j]:10.3f}' for j in range(len(cn))))
    pd.DataFrame(cm, index=cn, columns=cn).to_csv(OUTPUT_DIR / 'cm_fvi_raw_v2a.csv')
    pd.DataFrame(cm_norm, index=cn, columns=cn).to_csv(OUTPUT_DIR / 'cm_fvi_norm_v2a.csv')

# ============================================================
# PART G: TEST-RETEST RELIABILITY
# ============================================================
print('\n' + '=' * 70)
print('PART G: TEST-RETEST RELIABILITY')
print('=' * 70)

retest_results = []
for task in ['AVI', 'FVI', 'OVI']:
    n_cls = TASKS[task]['n_classes']
    for sub in subjects_eval:
        sessions = load_subject_task_by_session(DATASET_DIR, sub, task)
        if 'ses-01' not in sessions or 'ses-02' not in sessions: continue
        for ses_name, (p, im, lab, _) in sessions.items():
            if len(np.unique(lab)) < n_cls: continue
            res = run_tri_cv(p, im, lab, HP, n_cls, device, F1=8, D=2, n_splits=3)
            retest_results.append({'subject': sub, 'task': task, 'session': ses_name,
                'imagery_acc': res['img'], 'perception_acc': res['perc']})

retest_df = pd.DataFrame(retest_results)
retest_df.to_csv(OUTPUT_DIR / 'testretest_v2a.csv', index=False)

print('\nTest-retest correlations (Pearson r):')
for task in ['AVI', 'FVI', 'OVI']:
    t = retest_df[retest_df['task'] == task]
    s1 = t[t['session'] == 'ses-01'].set_index('subject')['imagery_acc']
    s2 = t[t['session'] == 'ses-02'].set_index('subject')['imagery_acc']
    common = s1.index.intersection(s2.index)
    if len(common) >= 5:
        r, p = stats.pearsonr(s1[common], s2[common])
        print(f'  {task}: r={r:.3f}, p={p:.4f} {sig_str(p)} (N={len(common)})')


PART F: CONFUSION MATRICES

Normalized confusion matrix:
                   circle   pentagram      square
  circle            0.349       0.347       0.304
  pentagram         0.280       0.418       0.301
  square            0.288       0.361       0.351

PART G: TEST-RETEST RELIABILITY

Test-retest correlations (Pearson r):
  AVI: r=0.146, p=0.5636 n.s. (N=18)
  FVI: r=0.235, p=0.3325 n.s. (N=19)
  OVI: r=-0.158, p=0.5306 n.s. (N=18)


In [14]:
# ============================================================
# PART H: FINAL COMPARISON AND STATISTICS
# All stats on 19 evaluation subjects only.
# ============================================================
print('=' * 80)
print(f'PART H: FINAL COMPARISON (N={len(subjects_eval)} eval subjects)')
print('=' * 80)

# Merge all results
tri_82 = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')}
                        for r in tri_results if r['arch'] == '8_2'])
tri_42 = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')}
                        for r in tri_results if r['arch'] == '4_2'])
fbcsp_eval = fbcsp_df[fbcsp_df['subject'].isin(subjects_eval)].copy()

abl_f = ablation_df[ablation_df['ablation'] == 'frontal_7'][['subject','task','abl_img','abl_perc']].rename(
    columns={'abl_img': 'frontal_img', 'abl_perc': 'frontal_perc'})
abl_p = ablation_df[ablation_df['ablation'] == 'posterior_only'][['subject','task','abl_img','abl_perc']].rename(
    columns={'abl_img': 'posterior_img', 'abl_perc': 'posterior_perc'})

merged = tri_82.merge(fbcsp_eval, on=['subject', 'task'])
merged = merged.merge(dense_ce_df, on=['subject', 'task'])
merged = merged.merge(abl_f, on=['subject', 'task'], how='left')
merged = merged.merge(abl_p, on=['subject', 'task'], how='left')
merged = merged.merge(tri_42[['subject','task','tri_img']].rename(columns={'tri_img': 'tri_42_img'}),
                      on=['subject', 'task'], how='left')

stat_results = []

for task in ['AVI', 'FVI', 'OVI']:
    chance = TASKS[task]['chance']
    t = merged[merged['task'] == task]
    print(f'\n{"="*70}')
    print(f'{task} (chance={chance:.3f}, N={len(t)})')
    print(f'{"="*70}')
    
    print(f'  {"Method":35s} {"Imagery":>12s}')
    for label, col in [('Tri-Contrastive (8,2)', 'tri_img'), ('Tri-Contrastive (4,2)', 'tri_42_img'),
                        ('Dense CE (8,2)', 'dense_ce_img'), ('FBCSP+LDA', 'lda_imagery'),
                        ('FBCSP+SVM', 'svm_imagery'), ('Ablated: frontal-7', 'frontal_img'),
                        ('Ablated: posterior-only', 'posterior_img')]:
        if col in t.columns and t[col].notna().any():
            v = t[col].dropna().values
            print(f'  {label:35s} {v.mean():.3f} +/- {v.std():.3f}')
    
    print(f'\n  Wilcoxon vs chance:')
    for name, col in [('Tri(8,2)', 'tri_img'), ('Tri(4,2)', 'tri_42_img'), ('Dense CE', 'dense_ce_img'),
                       ('FBCSP+LDA', 'lda_imagery'), ('FBCSP+SVM', 'svm_imagery'),
                       ('Frontal-7', 'frontal_img'), ('Posterior-only', 'posterior_img')]:
        v = t[col].dropna().values
        if len(v) < 3: continue
        try: W, p = stats.wilcoxon(v - chance, alternative='greater')
        except: W, p = 0, 1.0
        d = cohens_d(v - chance)
        print(f'    {name:30s}: {v.mean():.3f}  p={p:.4f} {sig_str(p):4s}  d={d:.3f}')
        stat_results.append({'task': task, 'test': 'vs_chance', 'method': name,
            'mean': v.mean(), 'std': v.std(), 'p': p, 'd': d, 'N': len(v)})
    
    print(f'\n  Paired comparisons:')
    tri_i = t['tri_img'].values
    for name, col_b in [('Tri(8,2) vs Dense CE', 'dense_ce_img'), ('Tri(8,2) vs FBCSP+LDA', 'lda_imagery'),
                         ('Tri(8,2) vs FBCSP+SVM', 'svm_imagery'), ('Tri(8,2) vs Frontal-7', 'frontal_img'),
                         ('Tri(8,2) vs Posterior', 'posterior_img')]:
        b = t[col_b].dropna().values
        a = tri_i[:len(b)]
        if len(a) < 3: continue
        try: W, p = stats.wilcoxon(a - b, alternative='greater')
        except: W, p = 0, 1.0
        d = cohens_d(a, b)
        print(f'    {name:35s}: d={a.mean()-b.mean():+.3f}  p={p:.4f} {sig_str(p):4s}  d={d:.3f}')
        stat_results.append({'task': task, 'test': 'paired', 'method': name,
            'delta': a.mean()-b.mean(), 'p': p, 'd': d, 'N': len(a)})

if len(cross_df) > 0:
    print(f'\n{"="*70}\nCROSS-SESSION GENERALIZATION\n{"="*70}')
    for task in ['AVI', 'FVI', 'OVI']:
        ct = cross_df[cross_df['task'] == task]
        if len(ct) == 0: continue
        v = ct['img_mean'].values
        chance = TASKS[task]['chance']
        try: W, p = stats.wilcoxon(v - chance, alternative='greater')
        except: W, p = 0, 1.0
        d = cohens_d(v - chance)
        print(f'  {task}: {v.mean():.3f} +/- {v.std():.3f}  p={p:.4f} {sig_str(p)}  d={d:.3f} (N={len(ct)})')
        stat_results.append({'task': task, 'test': 'cross_session', 'method': 'Tri(8,2)',
            'mean': v.mean(), 'std': v.std(), 'p': p, 'd': d, 'N': len(ct)})

print(f'\n{"="*80}\nPOOLED ACROSS ALL TASKS\n{"="*80}')
for col, name in [('tri_img','Tri(8,2)'), ('tri_42_img','Tri(4,2)'), ('dense_ce_img','Dense CE'),
                   ('lda_imagery','FBCSP+LDA'), ('svm_imagery','FBCSP+SVM')]:
    above = merged.apply(lambda r: r[col] - TASKS[r['task']]['chance'], axis=1).dropna().values
    try: W, p = stats.wilcoxon(above, alternative='greater')
    except: W, p = 0, 1.0
    d = cohens_d(above)
    print(f'  {name:25s} > chance: {above.mean():+.4f}  p={p:.6f} {sig_str(p):4s}  d={d:.3f}')
    stat_results.append({'task': 'POOLED', 'test': 'vs_chance', 'method': name,
        'mean_above_chance': above.mean(), 'p': p, 'd': d, 'N': len(above)})

# Save everything
merged.to_csv(OUTPUT_DIR / 'all_results_v2a.csv', index=False)
pd.DataFrame(stat_results).to_csv(OUTPUT_DIR / 'statistical_tests_v2a.csv', index=False)

with open(OUTPUT_DIR / 'run_config_v2a.json', 'w') as f:
    json.dump({
        'primary_architecture': 'EEGNet-8,2 (F1=8, D=2)',
        'comparison_architecture': 'EEGNet-4,2 (F1=4, D=2)',
        'hyperparameters': HP,
        'n_folds': N_FOLDS, 'crop_length': CROP_LENGTH,
        'search_subjects_EXCLUDED': SEARCH_SUBJECTS,
        'eval_subjects': subjects_eval,
        'fbcsp': '9 non-overlapping 4Hz bands (4-40Hz), SelectKBest(MI, k=18)',
        'dense_ce': 'Imagery-only, pure CE loss, NO perception/contrastive/prototypes',
        'ablations': {'frontal_7': FRONTAL_ABLATION_CHANNELS, 'posterior_only': POSTERIOR_RETAIN_CHANNELS},
        'cv': 'purged_temporal_block (N=5, purge_width=1)',
        'ica': 'cached, per-session, iclabel, eye_blink+muscle>0.80',
    }, f, indent=2, default=str)

print(f'\n{"="*80}')
print(f'ALL V2a RESULTS SAVED TO: {OUTPUT_DIR}')
print(f'{"="*80}')


PART H: FINAL COMPARISON (N=19 eval subjects)

AVI (chance=0.333, N=19)
  Method                                   Imagery
  Tri-Contrastive (8,2)               0.369 +/- 0.034
  Tri-Contrastive (4,2)               0.351 +/- 0.028
  Dense CE (8,2)                      0.365 +/- 0.034
  FBCSP+LDA                           0.340 +/- 0.046
  FBCSP+SVM                           0.336 +/- 0.047
  Ablated: frontal-7                  0.358 +/- 0.037
  Ablated: posterior-only             0.348 +/- 0.030

  Wilcoxon vs chance:
    Tri(8,2)                      : 0.369  p=0.0005 ***   d=1.067
    Tri(4,2)                      : 0.351  p=0.0111 *     d=0.632
    Dense CE                      : 0.365  p=0.0014 **    d=0.919
    FBCSP+LDA                     : 0.340  p=0.3217 n.s.  d=0.137
    FBCSP+SVM                     : 0.336  p=0.4566 n.s.  d=0.061
    Frontal-7                     : 0.358  p=0.0053 **    d=0.664
    Posterior-only                : 0.348  p=0.0370 *     d=0.478

  Paired comp

## Output Summary

| File | Description |
|------|-------------|
| `fbcsp_baselines_v2a.csv` | FBCSP+LDA/SVM baselines (all 22 subjects) |
| `best_hparams_v2a.json` | Optuna best hyperparameters |
| `optuna_study_v2a.csv` | Full Optuna trial history |
| `tri_results_v2a.csv` | Tri-contrastive results (8,2 and 4,2) |
| `dense_ce_results_v2a.csv` | Dense CE baseline results |
| `ablation_results_v2a.csv` | Both ablation conditions |
| `cross_session_v2a.csv` | Cross-session generalization |
| `cm_fvi_*.csv` | FVI confusion matrices |
| `testretest_v2a.csv` | Test-retest per session |
| `statistical_tests_v2a.csv` | All statistical tests |
| `all_results_v2a.csv` | Merged master table (19 eval subjects) |
| `run_config_v2a.json` | Full run configuration |

**Guardrails verified:**
1. FBCSP uses 9-band MI feature selection (not nerfed)
2. Search subjects are physically removed from `subjects_eval`
3. Dense CE uses NO perception, NO contrastive loss, NO prototypes


## V2B

In [22]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  V2b CELLS — Append these to the bottom of v2a_pipeline.ipynb              ║
# ║  Each "# %% [markdown]" or "# %%" marks a new cell.                       ║
# ║  Copy everything between cell markers into a new notebook cell.            ║
# ║  Prerequisites: all v2a cells have been run (helpers, data, subjects_eval) ║
# ╚══════════════════════════════════════════════════════════════════════════════╝


# %% [markdown]
# ---
# # V2b: Architecture Exploration
#
# **Goal**: Determine whether the contrastive objective can be rescued with
# (1) better hyperparameter optimization and (2) richer architectures.
#
# **What's new:**
# - Re-run Optuna with better search subjects (240 trials each), wider `lam_classify`, 5-fold
# - Separate Optuna for Dense CE (fair comparison)
# - Multi-scale temporal convolutions
# - Temporal attention pooling
# - All configs evaluated on the same 19 eval subjects from v2a
#
# **What we do NOT re-run:** ICA (cached), FBCSP baselines (saved), subject selection (same 19)


In [23]:
# ============================================================
# V2b CELL 1: Pick better Optuna search subjects
# Require 240 trials (both sessions), near-median FBCSP
# ============================================================

# Find subjects with data in both sessions for the search task
subjects_with_both = []
for sub in subjects_all:
    ses1 = DATASET_DIR / sub / 'ses-01' / 'eeg' / f'{sub}_ses-01_task-{SEARCH_TASK}_eeg.bdf'
    ses2 = DATASET_DIR / sub / 'ses-02' / 'eeg' / f'{sub}_ses-02_task-{SEARCH_TASK}_eeg.bdf'
    if ses1.exists() and ses2.exists():
        subjects_with_both.append(sub)

print(f'Subjects with both sessions for {SEARCH_TASK}: {len(subjects_with_both)}')

# Rank these by FBCSP/LDA imagery accuracy
fbcsp_both = fbcsp_df[
    (fbcsp_df['subject'].isin(subjects_with_both)) &
    (fbcsp_df['task'] == SEARCH_TASK)
].copy()
fbcsp_both = fbcsp_both.sort_values('lda_imagery').reset_index(drop=True)

print(f'\nFBCSP/LDA imagery for {SEARCH_TASK} (subjects with both sessions):')
for _, row in fbcsp_both.iterrows():
    marker = ' <-- eval excluded' if row['subject'] in SEARCH_SUBJECTS else ''
    print(f'  {row["subject"]}: {row["lda_imagery"]:.3f}{marker}')

# Pick 3 near 25th/50th/75th percentile from subjects NOT already excluded
eligible = fbcsp_both[~fbcsp_both['subject'].isin(SEARCH_SUBJECTS)].reset_index(drop=True)
n = len(eligible)
V2B_SEARCH_SUBJECTS = [
    eligible.iloc[int(n * 0.25)]['subject'],
    eligible.iloc[int(n * 0.50)]['subject'],
    eligible.iloc[int(n * 0.75)]['subject'],
]

print(f'\nV2b search subjects: {V2B_SEARCH_SUBJECTS}')
for s in V2B_SEARCH_SUBJECTS:
    row = eligible[eligible['subject'] == s].iloc[0]
    print(f'  {s}: FBCSP/LDA = {row["lda_imagery"]:.3f}')

# Updated eval list: exclude BOTH v2a and v2b search subjects
# subjects_eval_v2b = [s for s in subjects_all
#                      if s not in SEARCH_SUBJECTS and s not in V2B_SEARCH_SUBJECTS]
# print(f'\nV2b eval subjects ({len(subjects_eval_v2b)}): {subjects_eval_v2b}')

# If you prefer to keep the same 19 eval subjects from v2a and just use
# v2b search subjects that overlap with the original eval list, uncomment:
subjects_eval_v2b = [s for s in subjects_all if s not in V2B_SEARCH_SUBJECTS]
print(f'\nV2b eval subjects ({len(subjects_eval_v2b)}): {subjects_eval_v2b}')
# This would give you 16 eval subjects instead of 16.
# For now we exclude all search subjects for maximum rigor.

assert all(s not in subjects_eval_v2b for s in V2B_SEARCH_SUBJECTS)
# assert all(s not in subjects_eval_v2b for s in SEARCH_SUBJECTS)
print(f'All search subjects excluded from eval: VERIFIED')


Subjects with both sessions for FVI: 20

FBCSP/LDA imagery for FVI (subjects with both sessions):
  sub-08: 0.292
  sub-16: 0.296
  sub-11: 0.308
  sub-01: 0.312 <-- eval excluded
  sub-13: 0.325
  sub-20: 0.329
  sub-14: 0.333
  sub-04: 0.333
  sub-02: 0.333
  sub-15: 0.342
  sub-18: 0.350
  sub-06: 0.350
  sub-03: 0.362
  sub-19: 0.371
  sub-07: 0.375
  sub-05: 0.379
  sub-12: 0.388
  sub-17: 0.392
  sub-22: 0.400
  sub-21: 0.667

V2b search subjects: ['sub-20', 'sub-18', 'sub-05']
  sub-20: FBCSP/LDA = 0.329
  sub-18: FBCSP/LDA = 0.350
  sub-05: FBCSP/LDA = 0.379

V2b eval subjects (19): ['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10', 'sub-11', 'sub-12', 'sub-13', 'sub-14', 'sub-15', 'sub-16', 'sub-17', 'sub-19', 'sub-21', 'sub-22']
All search subjects excluded from eval: VERIFIED


In [24]:
# ============================================================
# V2b CELL 2: Optuna search — Tri-Contrastive with wider range
# Key change: lam_classify up to 8.0, 5-fold CV, 240-trial subjects
# ============================================================

hp_path_v2b = OUTPUT_DIR / 'best_hparams_v2b_tri.json'

print(f'Preloading v2b search data for {V2B_SEARCH_SUBJECTS} on {SEARCH_TASK}...')
v2b_search_data = {}
for sub in V2B_SEARCH_SUBJECTS:
    perc, img, labels, _ = load_subject_task_data(DATASET_DIR, sub, SEARCH_TASK)
    if perc is not None:
        v2b_search_data[sub] = (perc, img, labels)
        print(f'  {sub}: {len(labels)} trials')

v2b_search_splits = {sub: get_temporal_block_splits(labels, n_splits=5)
                     for sub, (_, _, labels) in v2b_search_data.items()}


def objective_v2b_tri(trial):
    hp = {
        'lr': trial.suggest_float('lr', 5e-5, 3e-3, log=True),
        'wd': trial.suggest_float('wd', 1e-3, 5e-2, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [16, 24, 32, 48]),
        'emb_dim': trial.suggest_categorical('emb_dim', [48, 64, 96, 128]),
        'hidden_dim': trial.suggest_categorical('hidden_dim', [96, 128, 192]),
        'temperature': trial.suggest_float('temperature', 0.03, 0.5, log=True),
        'w_img_perc': 1.0,
        'w_img_img': trial.suggest_float('w_img_img', 0.3, 3.0),
        'w_perc_img': trial.suggest_float('w_perc_img', 0.01, 0.5, log=True),
        'lam_within': trial.suggest_float('lam_within', 0.05, 2.0),
        'lam_align': trial.suggest_float('lam_align', 0.1, 3.0),
        'lam_classify': trial.suggest_float('lam_classify', 1.0, 8.0),  # wider range!
        'ch_dropout': trial.suggest_float('ch_dropout', 0.0, 0.25),
        'noise_scale': trial.suggest_float('noise_scale', 0.02, 0.25),
    }
    accs = []
    for sub, (perc, img, labels) in v2b_search_data.items():
        for tr, te in v2b_search_splits[sub]:
            trl, tel = make_tri_loaders(perc[tr], img[tr], labels[tr],
                                        perc[te], img[te], labels[te], hp)
            model = train_tri_model(trl, tel, hp, TASKS[SEARCH_TASK]['n_classes'],
                                    device, F1=8, D=2,
                                    n_epochs=SEARCH_EPOCHS, patience_limit=SEARCH_PATIENCE)
            ia, _, _, _ = probe_tri_model(model, trl, tel, device)
            accs.append(ia)
            trial.report(np.mean(accs), len(accs))
            if trial.should_prune():
                raise optuna.TrialPruned()
    return np.mean(accs)


if hp_path_v2b.exists():
    with open(hp_path_v2b) as f:
        bp_v2b_tri = json.load(f)
    print(f'Loaded cached v2b tri hparams')
else:
    study = optuna.create_study(direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=123),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5))
    print(f'Running {N_OPTUNA_TRIALS} Optuna trials (v2b tri-contrastive)...')
    t0 = time.time()
    study.optimize(objective_v2b_tri, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    print(f'Done in {(time.time()-t0)/60:.1f} min')
    print(f'Best trial: {study.best_trial.number}, best acc: {study.best_value:.4f}')
    bp_v2b_tri = study.best_params
    bp_v2b_tri['w_img_perc'] = 1.0
    with open(hp_path_v2b, 'w') as f:
        json.dump(bp_v2b_tri, f, indent=2)
    study.trials_dataframe().to_csv(OUTPUT_DIR / 'optuna_study_v2b_tri.csv', index=False)

HP_V2B_TRI = {**bp_v2b_tri, 'w_img_perc': 1.0}
print(f'\nV2b Tri-Contrastive HPs:')
for k, v in HP_V2B_TRI.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')


Preloading v2b search data for ['sub-20', 'sub-18', 'sub-05'] on FVI...
  sub-20: 240 trials
  sub-18: 240 trials
  sub-05: 240 trials
Running 50 Optuna trials (v2b tri-contrastive)...


Best trial: 23. Best value: 0.406944: 100%|██████████| 50/50 [53:44<00:00, 64.49s/it]

Done in 53.7 min
Best trial: 23, best acc: 0.4069

V2b Tri-Contrastive HPs:
  lr: 0.0003
  wd: 0.0057
  batch_size: 24
  emb_dim: 64
  hidden_dim: 96
  temperature: 0.3286
  w_img_img: 2.1029
  w_perc_img: 0.0227
  lam_within: 1.3211
  lam_align: 0.7872
  lam_classify: 3.7941
  ch_dropout: 0.1063
  noise_scale: 0.1502
  w_img_perc: 1.0000


In [25]:
# ============================================================
# V2b CELL 3: Optuna search — Dense CE (separate, fair)
# Smaller search space: no contrastive weights
# ============================================================

hp_path_v2b_ce = OUTPUT_DIR / 'best_hparams_v2b_ce.json'


def objective_v2b_ce(trial):
    hp = {
        'lr': trial.suggest_float('lr', 5e-5, 3e-3, log=True),
        'wd': trial.suggest_float('wd', 1e-3, 5e-2, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [16, 24, 32, 48]),
        'ch_dropout': trial.suggest_float('ch_dropout', 0.0, 0.25),
        'noise_scale': trial.suggest_float('noise_scale', 0.02, 0.25),
    }
    accs = []
    for sub, (_, img, labels) in v2b_search_data.items():
        for tr, te in v2b_search_splits[sub]:
            trl, tel = make_img_loaders(img[tr], labels[tr], img[te], labels[te], hp)
            model = train_dense_ce_model(trl, tel, hp, TASKS[SEARCH_TASK]['n_classes'],
                                         device, F1=8, D=2,
                                         n_epochs=SEARCH_EPOCHS, patience_limit=SEARCH_PATIENCE)
            acc, _, _ = probe_dense_ce(model, tel, device)
            accs.append(acc)
            trial.report(np.mean(accs), len(accs))
            if trial.should_prune():
                raise optuna.TrialPruned()
    return np.mean(accs)


if hp_path_v2b_ce.exists():
    with open(hp_path_v2b_ce) as f:
        bp_v2b_ce = json.load(f)
    print(f'Loaded cached v2b CE hparams')
else:
    study_ce = optuna.create_study(direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=456),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5))
    print(f'Running {N_OPTUNA_TRIALS} Optuna trials (v2b dense CE)...')
    t0 = time.time()
    study_ce.optimize(objective_v2b_ce, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    print(f'Done in {(time.time()-t0)/60:.1f} min')
    print(f'Best trial: {study_ce.best_trial.number}, best acc: {study_ce.best_value:.4f}')
    bp_v2b_ce = study_ce.best_params
    with open(hp_path_v2b_ce, 'w') as f:
        json.dump(bp_v2b_ce, f, indent=2)
    study_ce.trials_dataframe().to_csv(OUTPUT_DIR / 'optuna_study_v2b_ce.csv', index=False)

HP_V2B_CE = bp_v2b_ce
print(f'\nV2b Dense CE HPs:')
for k, v in HP_V2B_CE.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')


Running 50 Optuna trials (v2b dense CE)...


Best trial: 26. Best value: 0.411111: 100%|██████████| 50/50 [21:53<00:00, 26.27s/it]

Done in 21.9 min
Best trial: 26, best acc: 0.4111

V2b Dense CE HPs:
  lr: 0.0003
  wd: 0.0128
  batch_size: 16
  ch_dropout: 0.0755
  noise_scale: 0.1772


In [26]:
# ## V2b Architecture Variants
#
# New encoder variants. All use the same `EEGEncoder` API
# (input: `(B, 32, T)` + modality → output: `(B, F2+16)`) so they drop
# into `TriContrastiveModel` and `DenseCEModel` without changes to
# training/probing code.


# %%
# ============================================================
# V2b CELL 4: Multi-scale Temporal Encoder + Attention Pool
# ============================================================

class MultiScaleTemporalConv(nn.Module):
    """Parallel temporal convolutions at multiple kernel sizes."""
    def __init__(self, n_channels, F1=8, kernel_sizes=(16, 32, 64)):
        super().__init__()
        self.n_scales = len(kernel_sizes)
        # Distribute F1 filters across scales as evenly as possible
        filters_per_scale = [F1 // self.n_scales] * self.n_scales
        for i in range(F1 % self.n_scales):
            filters_per_scale[i] += 1
        self.filters_per_scale = filters_per_scale

        self.branches = nn.ModuleList()
        for n_f, ks in zip(filters_per_scale, kernel_sizes):
            self.branches.append(nn.Sequential(
                nn.Conv1d(n_channels, n_f, kernel_size=ks,
                          padding=ks // 2, bias=False),
                nn.BatchNorm1d(n_f),
            ))

    def forward(self, x):
        # x: (B, n_channels, T)
        outs = [branch(x) for branch in self.branches]
        return torch.cat(outs, dim=1)  # (B, F1, T)


class TemporalAttentionPool(nn.Module):
    """Learned attention-weighted temporal pooling."""
    def __init__(self, n_features):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(n_features, max(n_features // 4, 4)),
            nn.Tanh(),
            nn.Linear(max(n_features // 4, 4), 1))

    def forward(self, x):
        # x: (B, F, T)
        w = self.attn(x.transpose(1, 2))  # (B, T, 1)
        w = F.softmax(w, dim=1)            # (B, T, 1)
        return (x * w.transpose(1, 2)).sum(dim=2)  # (B, F)


class EEGEncoderV2b(nn.Module):
    """
    V2b encoder with optional multi-scale temporal and attention pooling.

    Args:
        n_channels: EEG channels (32)
        F1: total temporal filters (split across scales if multi_scale=True)
        D: spatial filters per temporal filter
        multi_scale: use parallel temporal convolutions at 3 scales
        attn_pool: use learned attention pooling instead of avg pool
        temporal_kernel_size: kernel size for single-scale (ignored if multi_scale)
        pool_size: spatial/separable pooling factor
        dropout: dropout rate
    """
    def __init__(self, n_channels=32, F1=8, D=2,
                 multi_scale=False, attn_pool=False,
                 temporal_kernel_size=64, pool_size=4, dropout=0.4):
        super().__init__()
        F2 = F1 * D
        self.F1, self.D, self.F2 = F1, D, F2
        self.multi_scale = multi_scale
        self.attn_pool = attn_pool

        # Block 1a: Temporal convolution (single or multi-scale)
        if multi_scale:
            self.temporal_conv = MultiScaleTemporalConv(
                n_channels, F1, kernel_sizes=(16, 32, 64))
        else:
            self.temporal_conv = nn.Sequential(
                nn.Conv1d(n_channels, F1, kernel_size=temporal_kernel_size,
                          padding=temporal_kernel_size // 2, bias=False),
                nn.BatchNorm1d(F1))

        # Block 1b: Depthwise spatial convolution
        self.spatial_conv = nn.Sequential(
            nn.Conv1d(F1, F2, kernel_size=1, groups=F1, bias=False),
            nn.BatchNorm1d(F2), nn.ELU(), nn.AvgPool1d(pool_size), nn.Dropout(dropout))

        # Channel attention
        self.channel_attention = ChannelAttention(F2)

        # Block 2: Separable convolution
        self.sep_conv1 = nn.Sequential(
            nn.Conv1d(F2, F2, kernel_size=16, padding=8, groups=F2, bias=False),
            nn.Conv1d(F2, F2, kernel_size=1, bias=False),
            nn.BatchNorm1d(F2), nn.ELU(), nn.AvgPool1d(pool_size), nn.Dropout(dropout))

        # Block 3: Second separable conv + pooling
        if attn_pool:
            self.sep_conv2 = nn.Sequential(
                nn.Conv1d(F2, F2, kernel_size=8, padding=4, groups=F2, bias=False),
                nn.Conv1d(F2, F2, kernel_size=1, bias=False),
                nn.BatchNorm1d(F2), nn.ELU(), nn.Dropout(dropout))
            self.pool = TemporalAttentionPool(F2)
        else:
            self.sep_conv2 = nn.Sequential(
                nn.Conv1d(F2, F2, kernel_size=8, padding=4, groups=F2, bias=False),
                nn.Conv1d(F2, F2, kernel_size=1, bias=False),
                nn.BatchNorm1d(F2), nn.ELU(), nn.AdaptiveAvgPool1d(1), nn.Dropout(dropout))
            self.pool = None

        self.modality_emb = nn.Embedding(2, 16)

    @property
    def output_dim(self):
        return self.F2 + 16

    def forward(self, x, modality):
        x = self.temporal_conv(x)
        x = self.spatial_conv(x)
        x = self.channel_attention(x)
        x = self.sep_conv1(x)
        x = self.sep_conv2(x)
        if self.pool is not None:
            x = self.pool(x)       # (B, F2) via attention
        else:
            x = x.squeeze(-1)      # (B, F2) via adaptive avg pool
        return torch.cat([x, self.modality_emb(modality)], dim=1)


# -- Drop-in model wrappers using V2b encoder --
class TriContrastiveModelV2b(nn.Module):
    def __init__(self, n_channels=32, n_classes=3, embedding_dim=64,
                 eeg_hidden_dim=128, F1=8, D=2, multi_scale=False, attn_pool=False):
        super().__init__()
        self.n_classes = n_classes
        self.image_encoder = FixedImageEncoder(n_classes, embedding_dim)
        self.eeg_encoder = EEGEncoderV2b(n_channels=n_channels, F1=F1, D=D,
                                          multi_scale=multi_scale, attn_pool=attn_pool)
        feat_dim = self.eeg_encoder.output_dim
        self.eeg_projection = ProjectionHead(feat_dim, eeg_hidden_dim, embedding_dim)
        self.classifier = ClassificationHead(feat_dim, n_classes)

    def forward(self, perception_eeg, imagery_eeg, labels):
        B, dev = labels.shape[0], labels.device
        image_emb = self.image_encoder(labels)
        perc_feat = self.eeg_encoder(perception_eeg, torch.zeros(B, dtype=torch.long, device=dev))
        perc_emb, perc_logits = self.eeg_projection(perc_feat), self.classifier(perc_feat)
        img_feat = self.eeg_encoder(imagery_eeg, torch.ones(B, dtype=torch.long, device=dev))
        img_emb, img_logits = self.eeg_projection(img_feat), self.classifier(img_feat)
        return image_emb, perc_emb, img_emb, perc_logits, img_logits


class DenseCEModelV2b(nn.Module):
    def __init__(self, n_channels=32, n_classes=3, F1=8, D=2,
                 multi_scale=False, attn_pool=False):
        super().__init__()
        self.eeg_encoder = EEGEncoderV2b(n_channels=n_channels, F1=F1, D=D,
                                          multi_scale=multi_scale, attn_pool=attn_pool)
        self.classifier = ClassificationHead(self.eeg_encoder.output_dim, n_classes)

    def forward(self, imagery_eeg):
        B, dev = imagery_eeg.shape[0], imagery_eeg.device
        feat = self.eeg_encoder(imagery_eeg, torch.ones(B, dtype=torch.long, device=dev))
        return self.classifier(feat)


# -- V2b training wrappers (use V2b models, otherwise identical) --
def train_tri_model_v2b(train_loader, test_loader, hp, n_classes, device,
                        F1=8, D=2, multi_scale=False, attn_pool=False,
                        n_epochs=150, patience_limit=25):
    model = TriContrastiveModelV2b(n_channels=32, n_classes=n_classes,
        embedding_dim=hp.get('emb_dim', 64), eeg_hidden_dim=hp.get('hidden_dim', 128),
        F1=F1, D=D, multi_scale=multi_scale, attn_pool=attn_pool).to(device)
    criterion = TriContrastiveLoss(
        temperature=hp.get('temperature', 0.1), w_image_perc=hp.get('w_img_perc', 1.0),
        w_image_img=hp.get('w_img_img', 1.0), w_perc_img=hp.get('w_perc_img', 0.2),
        lambda_within=hp.get('lam_within', 0.5), lambda_align=hp.get('lam_align', 1.0),
        lambda_classify=hp.get('lam_classify', 2.0))
    def crit_fn(mdl, batch):
        p_b, i_b, l_b = batch
        ie, pe, ime, pl, il = mdl(p_b, i_b, l_b)
        loss, _ = criterion(ie, pe, ime, l_b, pl, il)
        return loss
    opt = torch.optim.AdamW(model.parameters(), lr=hp.get('lr', 5e-4), weight_decay=hp.get('wd', 1e-2))
    warmup = LinearLR(opt, start_factor=0.01, total_iters=10)
    cosine = CosineAnnealingLR(opt, T_max=n_epochs - 10, eta_min=1e-6)
    sched = SequentialLR(opt, [warmup, cosine], milestones=[10])
    return _train_loop(model, crit_fn, train_loader, test_loader, opt, sched,
                       n_epochs, patience_limit, device)


def train_dense_ce_v2b(train_loader, test_loader, hp, n_classes, device,
                       F1=8, D=2, multi_scale=False, attn_pool=False,
                       n_epochs=150, patience_limit=25):
    model = DenseCEModelV2b(n_channels=32, n_classes=n_classes, F1=F1, D=D,
                             multi_scale=multi_scale, attn_pool=attn_pool).to(device)
    ce = nn.CrossEntropyLoss()
    def crit_fn(mdl, batch):
        img_b, lab_b = batch
        return ce(mdl(img_b), lab_b)
    opt = torch.optim.AdamW(model.parameters(), lr=hp.get('lr', 5e-4), weight_decay=hp.get('wd', 1e-2))
    warmup = LinearLR(opt, start_factor=0.01, total_iters=10)
    cosine = CosineAnnealingLR(opt, T_max=n_epochs - 10, eta_min=1e-6)
    sched = SequentialLR(opt, [warmup, cosine], milestones=[10])
    return _train_loop(model, crit_fn, train_loader, test_loader, opt, sched,
                       n_epochs, patience_limit, device)


# -- V2b CV runners --
def run_tri_cv_v2b(perc, img, labels, hp, n_cls, device,
                   F1=8, D=2, multi_scale=False, attn_pool=False, n_splits=5):
    splits = get_temporal_block_splits(labels, n_splits)
    img_accs, perc_accs = [], []
    for tr, te in splits:
        trl, tel = make_tri_loaders(perc[tr], img[tr], labels[tr],
                                    perc[te], img[te], labels[te], hp)
        model = train_tri_model_v2b(trl, tel, hp, n_cls, device, F1=F1, D=D,
                                     multi_scale=multi_scale, attn_pool=attn_pool)
        ia, pa, _, _ = probe_tri_model(model, trl, tel, device)
        img_accs.append(ia); perc_accs.append(pa)
    return {'img': np.mean(img_accs), 'perc': np.mean(perc_accs),
            'img_folds': img_accs, 'perc_folds': perc_accs}


def run_dense_ce_cv_v2b(img, labels, hp, n_cls, device,
                        F1=8, D=2, multi_scale=False, attn_pool=False, n_splits=5):
    splits = get_temporal_block_splits(labels, n_splits)
    img_accs = []
    for tr, te in splits:
        trl, tel = make_img_loaders(img[tr], labels[tr], img[te], labels[te], hp)
        model = train_dense_ce_v2b(trl, tel, hp, n_cls, device, F1=F1, D=D,
                                    multi_scale=multi_scale, attn_pool=attn_pool)
        acc, _, _ = probe_dense_ce(model, tel, device)
        img_accs.append(acc)
    return {'img': np.mean(img_accs), 'img_folds': img_accs}


# Parameter counts for all configs
print('Parameter counts:')
for name, ms, ap in [('baseline (8,2)', False, False),
                      ('multi-scale (8,2)', True, False),
                      ('multi-scale+attn (8,2)', True, True)]:
    m = TriContrastiveModelV2b(n_channels=32, n_classes=3, F1=8, D=2,
                                multi_scale=ms, attn_pool=ap)
    print(f'  {name}: {sum(p.numel() for p in m.parameters()):,} params')


Parameter counts:
  baseline (8,2): 49,399 params
  multi-scale (8,2): 41,719 params
  multi-scale+attn (8,2): 41,792 params


In [27]:
# ============================================================
# V2b CELL 5: Run all architecture configs
# Quick eval: 5-fold CV on eval subjects, imagery accuracy only
# ============================================================

CONFIGS = [
    # (tag, loss_type, multi_scale, attn_pool, hp_dict)
    ('tri_8_2_v2b',          'tri',  False, False, HP_V2B_TRI),
    ('ce_8_2_v2b',           'ce',   False, False, HP_V2B_CE),
    ('tri_ms_8_2',           'tri',  True,  False, HP_V2B_TRI),
    ('ce_ms_8_2',            'ce',   True,  False, HP_V2B_CE),
    ('tri_ms_attn_8_2',      'tri',  True,  True,  HP_V2B_TRI),
    ('ce_ms_attn_8_2',       'ce',   True,  True,  HP_V2B_CE),
]

print('=' * 80)
print(f'V2b ARCHITECTURE EXPLORATION ({len(subjects_eval_v2b)} eval subjects)')
print('=' * 80)
print(f'\nConfigs to run:')
for tag, lt, ms, ap, _ in CONFIGS:
    print(f'  {tag}: loss={lt}, multi_scale={ms}, attn_pool={ap}')

v2b_results = []

for tag, loss_type, ms, ap, hp in CONFIGS:
    print(f'\n{"="*70}')
    print(f'Config: {tag}')
    print(f'{"="*70}')

    for task in ['AVI', 'FVI', 'OVI']:
        n_cls = TASKS[task]['n_classes']
        print(f'\n--- {task} ---')
        t0 = time.time()

        for i, sub in enumerate(subjects_eval_v2b):
            perc, img, labels, _ = load_subject_task_data(DATASET_DIR, sub, task)
            if perc is None or len(np.unique(labels)) < n_cls:
                print(f'  [{i+1}/{len(subjects_eval_v2b)}] {sub}: SKIP')
                continue

            if loss_type == 'tri':
                res = run_tri_cv_v2b(perc, img, labels, hp, n_cls, device,
                                     F1=8, D=2, multi_scale=ms, attn_pool=ap)
                v2b_results.append({
                    'subject': sub, 'task': task, 'config': tag,
                    'img_acc': res['img'], 'perc_acc': res['perc'],
                    'img_folds': res['img_folds']})
            else:
                res = run_dense_ce_cv_v2b(img, labels, hp, n_cls, device,
                                          F1=8, D=2, multi_scale=ms, attn_pool=ap)
                v2b_results.append({
                    'subject': sub, 'task': task, 'config': tag,
                    'img_acc': res['img'], 'perc_acc': None,
                    'img_folds': res['img_folds']})

            print(f'  [{i+1}/{len(subjects_eval_v2b)}] {sub}: I:{res["img"]:.3f}')

        print(f'  [{task} done in {(time.time()-t0)/60:.1f} min]')

v2b_df = pd.DataFrame(v2b_results)
v2b_df.to_csv(OUTPUT_DIR / 'v2b_architecture_results.csv', index=False)
print(f'\nSaved {len(v2b_df)} results')



V2b ARCHITECTURE EXPLORATION (19 eval subjects)

Configs to run:
  tri_8_2_v2b: loss=tri, multi_scale=False, attn_pool=False
  ce_8_2_v2b: loss=ce, multi_scale=False, attn_pool=False
  tri_ms_8_2: loss=tri, multi_scale=True, attn_pool=False
  ce_ms_8_2: loss=ce, multi_scale=True, attn_pool=False
  tri_ms_attn_8_2: loss=tri, multi_scale=True, attn_pool=True
  ce_ms_attn_8_2: loss=ce, multi_scale=True, attn_pool=True

Config: tri_8_2_v2b

--- AVI ---
  [1/19] sub-01: I:0.358
  [2/19] sub-02: I:0.367
  [3/19] sub-03: I:0.379
  [4/19] sub-04: I:0.304
  [5/19] sub-06: I:0.350
  [6/19] sub-07: I:0.375
  [7/19] sub-08: I:0.317
  [8/19] sub-09: I:0.342
  [9/19] sub-10: I:0.283
  [10/19] sub-11: I:0.329
  [11/19] sub-12: I:0.396
  [12/19] sub-13: I:0.333
  [13/19] sub-14: I:0.321
  [14/19] sub-15: I:0.371
  [15/19] sub-16: I:0.354
  [16/19] sub-17: I:0.325
  [17/19] sub-19: I:0.367
  [18/19] sub-21: I:0.371
  [19/19] sub-22: I:0.417
  [AVI done in 10.0 min]

--- FVI ---
  [1/19] sub-01: I:0.346

In [28]:
# ============================================================
# V2b CELL 6: Summary comparison
# ============================================================

print('=' * 80)
print('V2b ARCHITECTURE COMPARISON')
print('=' * 80)

# Also include v2a results for reference (filter to same eval subjects)
print('\n--- V2a reference (from saved CSVs, filtered to v2b eval subjects) ---')
for ref_name, ref_path, col in [
    ('v2a Tri(8,2)', OUTPUT_DIR / 'tri_results_v2a.csv', 'tri_img'),
    ('v2a Dense CE', OUTPUT_DIR / 'dense_ce_results_v2a.csv', 'dense_ce_img'),
    ('v2a FBCSP+LDA', OUTPUT_DIR / 'fbcsp_baselines_v2a.csv', 'lda_imagery'),
]:
    if ref_path.exists():
        df = pd.read_csv(ref_path)
        if 'arch' in df.columns:
            df = df[df['arch'] == '8_2']
        df = df[df['subject'].isin(subjects_eval_v2b)]
        for task in ['AVI', 'FVI', 'OVI']:
            vals = df[df['task'] == task][col].dropna().values
            if len(vals) > 0:
                chance = TASKS[task]['chance']
                above = vals - chance
                try: _, p = stats.wilcoxon(above, alternative='greater')
                except: p = 1.0
                d = cohens_d(above)
                print(f'  {ref_name:25s} {task}: {vals.mean():.3f} +/- {vals.std():.3f}  '
                      f'p={p:.4f} {sig_str(p):4s}  d={d:.3f}')

print('\n--- V2b new configs ---')
summary_rows = []
for tag in [c[0] for c in CONFIGS]:
    for task in ['AVI', 'FVI', 'OVI']:
        chance = TASKS[task]['chance']
        vals = v2b_df[(v2b_df['config'] == tag) & (v2b_df['task'] == task)]['img_acc'].values
        if len(vals) < 3:
            continue
        above = vals - chance
        try:
            _, p = stats.wilcoxon(above, alternative='greater')
        except:
            p = 1.0
        d = cohens_d(above)
        print(f'  {tag:25s} {task}: {vals.mean():.3f} +/- {vals.std():.3f}  '
              f'p={p:.4f} {sig_str(p):4s}  d={d:.3f}')
        summary_rows.append({
            'config': tag, 'task': task,
            'mean': vals.mean(), 'std': vals.std(),
            'p': p, 'd': d, 'N': len(vals)})

# Pooled comparison
print(f'\n--- Pooled across tasks ---')
for tag in [c[0] for c in CONFIGS]:
    sub_df = v2b_df[v2b_df['config'] == tag].copy()
    above = sub_df.apply(lambda r: r['img_acc'] - TASKS[r['task']]['chance'], axis=1).values
    try:
        _, p = stats.wilcoxon(above, alternative='greater')
    except:
        p = 1.0
    d = cohens_d(above)
    print(f'  {tag:25s} pooled: {above.mean():+.4f} above chance  '
          f'p={p:.6f} {sig_str(p):4s}  d={d:.3f}')
    summary_rows.append({
        'config': tag, 'task': 'POOLED',
        'mean_above_chance': above.mean(), 'p': p, 'd': d, 'N': len(above)})

pd.DataFrame(summary_rows).to_csv(OUTPUT_DIR / 'v2b_summary.csv', index=False)

# Pairwise: does any tri config beat its CE counterpart?
print(f'\n--- Tri vs CE pairwise (same architecture) ---')
for tri_tag, ce_tag in [('tri_8_2_v2b', 'ce_8_2_v2b'),
                         ('tri_ms_8_2', 'ce_ms_8_2'),
                         ('tri_ms_attn_8_2', 'ce_ms_attn_8_2')]:
    for task in ['AVI', 'FVI', 'OVI']:
        tri_df_t = v2b_df[(v2b_df['config'] == tri_tag) & (v2b_df['task'] == task)]
        ce_df_t = v2b_df[(v2b_df['config'] == ce_tag) & (v2b_df['task'] == task)]
        # Merge on subject
        m = tri_df_t[['subject', 'img_acc']].merge(
            ce_df_t[['subject', 'img_acc']], on='subject', suffixes=('_tri', '_ce'))
        if len(m) < 3:
            continue
        diff = m['img_acc_tri'].values - m['img_acc_ce'].values
        try:
            _, p = stats.wilcoxon(diff, alternative='greater')
        except:
            p = 1.0
        d = cohens_d(diff)
        print(f'  {tri_tag} vs {ce_tag} [{task}]: '
              f'tri={m["img_acc_tri"].mean():.3f} ce={m["img_acc_ce"].mean():.3f} '
              f'delta={diff.mean():+.3f} p={p:.4f} {sig_str(p):4s}')

print(f'\n{"="*80}')
print('V2b exploration complete. Review results above to decide which')
print('config(s) merit full validation (ablation, cross-session, etc.)')
print(f'{"="*80}')

V2b ARCHITECTURE COMPARISON

--- V2a reference (from saved CSVs, filtered to v2b eval subjects) ---
  v2a Tri(8,2)              AVI: 0.372 +/- 0.029  p=0.0005 ***   d=1.316
  v2a Tri(8,2)              FVI: 0.377 +/- 0.078  p=0.0027 **    d=0.564
  v2a Tri(8,2)              OVI: 0.265 +/- 0.037  p=0.0978 n.s.  d=0.402
  v2a Dense CE              AVI: 0.361 +/- 0.033  p=0.0038 **    d=0.853
  v2a Dense CE              FVI: 0.394 +/- 0.054  p=0.0003 ***   d=1.136
  v2a Dense CE              OVI: 0.277 +/- 0.036  p=0.0019 **    d=0.744
  v2a FBCSP+LDA             AVI: 0.346 +/- 0.043  p=0.1136 n.s.  d=0.307
  v2a FBCSP+LDA             FVI: 0.365 +/- 0.078  p=0.0247 *     d=0.407
  v2a FBCSP+LDA             OVI: 0.250 +/- 0.031  p=0.6607 n.s.  d=0.016

--- V2b new configs ---
  tri_8_2_v2b               AVI: 0.350 +/- 0.032  p=0.0180 *     d=0.535
  tri_8_2_v2b               FVI: 0.380 +/- 0.073  p=0.0009 ***   d=0.645
  tri_8_2_v2b               OVI: 0.268 +/- 0.040  p=0.0371 *     d=0.453

In [29]:
# %% [markdown]
# ## What to do next
#
# Based on the v2b summary above:
#
# 1. **If a tri-contrastive config beats its CE counterpart**: The contrastive
#    objective is rescued by the architectural improvement. Run full validation
#    (ablation, cross-session, test-retest) on that config.
#
# 2. **If CE still wins across all architectures**: The contrastive objective
#    does not help for imagery classification with this dataset. The paper
#    story becomes "CNN encoder is sufficient; contrastive alignment provides
#    a shared embedding space but doesn't improve classification."
#
# 3. **If multi-scale or attention pool helps both tri and CE equally**: The
#    architectural improvement is orthogonal to the loss function. Report
#    the best CE model as your primary result, with the contrastive variant
#    as a secondary analysis.